In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Прочитаем все данные и выведем первые строки для наглядности


In [ ]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
submission = pd.read_csv('/kaggle/input/competitions/titanic/gender_submission.csv')

In [ ]:
train.head()
#test.head()
#submission.head()

In [ ]:
train.shape
#test.shape

Теперь необходимо подготовить наши данные для обучения модели. Начинаем с разбиения тренировочных данных на две группы: 
* тренировочная. На этих данных наша модель будет тренироваться
* и проверочные. Это данные, которые будут использоваться для оценивания точности предсказания нашей модели

Обычно 80% - это тренировочная часть и 20% - проверочная

In [ ]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['PassengerId', 'Survived'])
y = train['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

Теперь необходимо разобраться с пропавшими данными по каким-то причинам. Сначала необходимо узнать какие столбы вообще имеют неизвестные значения и сколько их.

In [ ]:
X_train.isna().sum() #вывод: все столбцы и сколько в них пропускох (считая ноль)
X_NaN = X_train.isna().sum()[X_train.isna().sum() > 0] #ноль не интересен, поэтому убираем его
X_NaN.head()

Воспользуемся методами избавления от неизвестных данных.
**Важно!** Необходимо знать ответ на вопрос: *Это значение отсутствует, потому что оно не было записано или потому что оно не существует?*

Если значение отсутствует, потому что оно не существует (например, рост старшего ребенка у кого-то, у кого нет детей), то нет смысла пытаться угадать, что это может быть. Эти значения, вероятно, действительно необходимо сохранить как NaN. 

Иначе, если значение отсутствует из-за того, что оно не было записано, можно попытаться угадать, что это могло быть

**Первый способ.** Удаление всех столбцов в которых есть значение NaN. Или удалить все строки со значением NaN. У этого способа есть огромный минус в виде потери значимого количества данных. В нашем случае либо потерей 3 столбцов из 12, либо потерей минимум 547 строк из 891, что очень значительно и в первом и во втором случае

In [ ]:
X_train_clean_col = X_train.dropna(axis=1)
#X_train_clean_col = X_train.drop(columns=X_NaN.index) - работает аналогично 

X_train_clean_row = X_train.dropna(axis=0)

In [ ]:
X_train_clean_col.head()

In [ ]:
X_train_clean_row.head()

**Второй способ.** Найти и заполнить все ячейки каким-то константным значением. Здесь очень важно всегда проверять что изменяемые данные - это наш тренировочный вариант, чтобы не произошло утечки данных (например, если использовать этот способ к dataframe X и заменять все числовые значения на средние, то произойдет утечка данных при обучении модели на X_train).

In [ ]:
X_train_replace = X_train.copy()

# Находим числовые столбцы и заполняем их средним
num_cols = X_train.select_dtypes(include='number').columns
X_train_replace[num_cols] = X_train[num_cols].fillna(X_train[num_cols].mean())

# Находим текстовые столбцы и заполняем их строкой 'unknown'
cat_cols = X_train.select_dtypes(include='object').columns
X_train_replace[cat_cols] = X_train[cat_cols].fillna('unknown')

In [ ]:
X_train_replace.head()

**Третий способ.** Последним вариантом является добавление нового столбца-индекса неизвестности. То есть это просто столбец который показывает, пропущен ли в каком-то из столбцов значение.

In [ ]:
X_train_add_ind = X_train.copy()
for col in X_NaN.index:
    # Создаем новый бинарный столбец-индикатор (1 - был пропуск, 0 - данные на месте)
    X_train_add_ind[f'{col}_isNaN'] = X_train[col].isna().astype(int)

In [ ]:
X_train_add_ind.head()

Теперь все те же операции необходимо проделать и для X_val ведь модель у нас не будет понимать что делать с данными NaN

In [ ]:
# 1 способ 
X_val_clean_col = X_val.dropna(axis=1)
X_val_clean_row = X_val.dropna(axis=0)

# 2 способ
X_val_replace = X_val.copy()
X_val_replace[num_cols] = X_val[num_cols].fillna(X_val[num_cols].mean())
X_val_replace[cat_cols] = X_val[cat_cols].fillna('unknown')

# 3 способ
X_val_add_ind = X_val.copy()
for col in X_NaN.index:
    # Создаем новый бинарный столбец-индикатор (1 - был пропуск, 0 - данные на месте)
    X_val_add_ind[f'{col}_isNaN'] = X_val[col].isna().astype(int)

После того как все неизвестные данные были обработанны каким-то из пердшествующих методов, необходимо дальше подготовить данные для модели, а именно подготовить числовые столбцы и категориальные для дальнейшей работы.


Сначала обработаем все числовые столбцы. Всего можно выделить 9 видов работы с числовыми признаками:
1. Анализ распределения.
2. Обработка выбросов.
3. Трасформация формы.
4. Масштабирование.
5. Мультиколлинеарность.
6. Снижение рамерности.
7. Отбор признаков
8. Бинаризация или дискретизация
9. Feature Engineering



**Шаг первый.** В первую очередь проанализируем все распределения:

Для этого будем использовать два параметра Skewness - коэффицент ассиметрии, то есть то на сколько данные склоняются в какую-то сторону. Если это значение будет >1 значит нам надо будет что-то делать с этими данными на последующих шагах. Вторым нашим параметром будет Kurtosis - число, которое показывает "остроту" пика и толщину хвостов распределения по сравнению с нормальными. Это коэффициент эксцесс. Если он высокий, то в данных есть очень резкий, высокий пик в одном месте (например, почти все значения сосредоточены вокруг одного числа) и при этом могут юыть очень длинные редкие хвосты с выбросами.

Для простоты будем использовать только X_train_replace, так как обычно такой подход приносить более хороший результат и не сильно загржает таблицу новыми данными. Первый способ нам сильно сократил строки. Третий способ для последующих обработок прийдется немного переделать и заменить все NaN данные на какую-то константу. Алгоритмы не умеют работать с NaN данными, поэтому у нас будет таблица второго способа с дополнительными столбцами-индикаторами. И обрабатывать столбцы-индикаторы мы не будем, поэтому для удобства третий способ так же отложим.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Cчитаем Skewness и Kurtosis для каждого столбца
for col in num_cols:
    skew_val = X_train_replace[col].skew()
    kurt_val = X_train_replace[col].kurt()
    
    print(f"Столбец: {col}")
    print(f"  -> Skewness (асимметрия): {skew_val:.2f}")
    print(f"  -> Kurtosis (эксцесс):    {kurt_val:.2f}")
    print("-" * 40)

# Визуальный анализ: строим гистограмму и Q-Q Plot для каждой числовой колонки
for col in num_cols:
    # Позволяет вывести два графика в одну строчку
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Слева: Гистограмма распределения + линия плотности (Density plot)
    sns.histplot(X_train_replace[col], kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'Гистограмма и плотность: {col}')
    
    # Справа: График Q-Q Plot
    stats.probplot(X_train_replace[col], dist="norm", plot=axes[1])
    axes[1].set_title(f'Q-Q Plot: {col}')
    
    plt.tight_layout()
    plt.show()

Гистограмма нам позволяет увидеть вокруг каких значений группируются данные и насколько плавно они распределены. Идеальные данные будут похожи на симметричную горку. Сразу видно что на последних трех графиках все данные сконцентрированны в начале: виден резкий пик и дальнейший тянущийся хвост. Это так же понятно по огромному значению kurtisis, но на графиках наглядно видно какие именно пики существуют. 

Второй график Q-Q Plot акентирует внимание на краях, ведь гистограмма часто их размывает. Если выбросов очень малое количество гистограмма их вообще не покажет. Поэтому второй график оценивает каество хвостов. Он берет квантильи реальных данных (синие точки) и сравнивает их с квантилями идеального нормального распределения (обозначено красной прямой линией). Поэтому если данные идельны, то они будут лежать строго на красной линии. Тога по графикам видно что сильные выбросы есть только у последних трех столбцов (правый хвост резко взлетает вверх, аналогично было бы если левые точки резко падали вниз)

**Шаг второй.** Мы примерно поняли какие столбцы имеют проблему. Отложим пока проблемные гистограмы и вторым шагом исправим все выбросы. Мы уже поняли что столбцы SibSp, Parch и Fare имеют выбросы. На данном шаге нам необходимо подлечить эти числовые столбцы.

Тут возникает сложность в выборе алгоритма. Существует 6 самых распространенных методов. Разберем подробно каждый способ. 

**Первый способ.** Рассмотрим самый распространенный способ работы с выбросами - IQR. Метод межквартильного размаха находит центральные 50% данных от 25-го до 75-го прецентиля. Размах между ними - это $IQR = Q_3 - Q_1$. Границы нормы: $[Q_1 - 1.5 * IQR, Q_3 + 1.5 * IQR]$. Значения же за границами не удаляются, а прижимаются к ним через метод .clip(). Этот метод устойчив к форме распределения (не ребует нормальности), поэтому он очень удобный и самый распространенный.

In [ ]:
X_train_replace_iqr = X_train_replace.copy()
X_val_replace_iqr = X_val_replace.copy()

for col in X_train_replace.select_dtypes(include='number').columns:
    q25, q75 = X_train_replace[col].quantile(0.25), X_train_replace[col].quantile(0.75)
    iqr = q75 - q25
    lower, upper = q25 - 1.5 * iqr, q75 + 1.5 * iqr
    
    X_train_replace_iqr[col] = X_train_replace_iqr[col].clip(lower=lower, upper=upper)
    X_val_replace_iqr[col] = X_val_replace_iqr[col].clip(lower=lower, upper=upper)

**Второй способ.** Вторым способом является метод трех сигм. Он работает строго по правилу нормального распределения, поэтому очень часто его невозможно применить. В пределах трех стандартных отклонений ($\sigma$) от среднего арифметического ($\mu$) лежит 99.7% всех данных. Границы нормы: $[\mu - 3\sigma, \; \mu + 3\sigma]$. Как уже и было сказано такой метод подходит только для признаков, которые на первом шаге показали форму симметричного колокола (например, Age). Если же применить его к сильно скошенным тратам, он посчитает неадекватные границы (нижняя граница уйдет в минус). 

In [ ]:
X_train_replace_3sig = X_train_replace.copy()
X_val_replace_3sig = X_val_replace.copy()

for col in X_train_replace.select_dtypes(include='number').columns:
    mu = X_train_replace[col].mean()
    sigma = X_train_replace[col].std()
    lower, upper = mu - 3 * sigma, mu + 3 * sigma
    
    X_train_replace_3sig[col] = X_train_replace_3sig[col].clip(lower=lower, upper=upper)
    X_val_replace_3sig[col] = X_val_replace_3sig[col].clip(lower=lower, upper=upper)

**Третий способ.** Третий способ, наверное, самый простой и понятный - это удаление по жестким прецентилям (Percentile Trimming). Мы жестко срезаем фиксированный процент самых экстремальных данных. Этот метод отлично подходит, когда нужно гарантированно избавиться от шума на краях, не привязываясь к средним или медианам. 

In [ ]:
X_train_replace_perc = X_train_replace.copy()
X_val_replace_perc = X_val.copy()

for col in X_train_replace.select_dtypes(include='number').columns:
    lower = X_train_replace[col].quantile(0.01)
    upper = X_train_replace[col].quantile(0.99)
    
    X_train_replace_perc[col] = X_train_replace_perc[col].clip(lower=lower, upper=upper)
    X_val_replace_perc[col] = X_val_replace_perc[col].clip(lower=lower, upper=upper)

Если до этого мы работали строго со столбцами, то четвертый и пятый способы анализируют уже всю таблицу. Они находят строки-аномалии по комбинации признаков и выдают метки 1 - нормальный обьект или -1 - выброс. **Важно,** что удалять строки по этим меткам можно только в тренировочных данных, в валидации удалять строки нельзя, чтобы не нарушить тестовую структуру ответов.

Эти два метода лучше всего использовать на уже немного линейно переработанных данных алгоритмами, описанными выше. Многомерные алгоритмы ищют нестыковку не в одном столбце (человек 200 лет), а в связи нескольких столбцов (например, пятилетний ребенок потратил несколько миллиардов долларов). Поэтому, чтобы не заставлять алгоритмы тратить большое количество времени на отделение этих столбцовых выбросов, предвартиельно их обрабатывают одномерными методами.

**Четвертый способ.** Изоляционный лес (Isolation Forest). Этот алгоритм строит ансамбль случайных деревьев и пытается изолировать каждый обьект случайными сплитами. Его логика определения выброса заключается в том, что если точка стоит в куче с другими, дереву нужно много раз ветвиться, чтобы до нее добраться. Если точка одинокая и странная, то она изолируется за 2-3 шага.

In [ ]:
from sklearn.ensemble import IsolationForest

# contamination=0.05 означает, что мы ожидаем около 5% выбросов в данных
iso_forest = IsolationForest(contamination=0.05, random_state=1) 

# Находим аномалии
train_labels = iso_forest.fit_predict(X_train_replace_iqr.select_dtypes(include='number'))

# Удаляем аномальные СТРОКИ только из тренировочного набора
X_train_replace_no_outliers = X_train_replace_iqr[train_labels == 1]
y_train_no_outliers = y_train[train_labels == 1] # не забываем отфильтровать таргет!

**Пятый способ.** Следующим методом является Local Outlier Factor (LOF). Алгоритм оценивает локальную плотность обьектов. Он берет точку, смотрит на ее k-ближайших соседей и сравнивает их плотности. Если точка лежит в разреженном пространстве, а все ее соседи - в плотном кластере, значит, эта точка - выброс.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# Обучаем LOF. Параметр novelty=True обязателен, чтобы метод мог предсказывать на новых данных
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True)
lof.fit(X_train_replace_iqr.select_dtypes(include='number'))

train_labels_lof = lof.predict(X_train_replace_iqr.select_dtypes(include='number'))

X_train_replace_lof_clean = X_train_replace_iqr[train_labels_lof == 1]
y_train_lof_clean = y_train[train_labels_lof == 1]

**Шестой способ.** Последним способом является лечение скейлингом. То есть мы пытаемся избавиться от выбросов не их удалением, а каким-то образом масштабированием. Для этого на шаге четыре нам понадобиться применять RobustScale, который центрирует данные по медиане и масштабирует их по IQR, благодаря чему выбросы сжимаются математически и не сдвигают веса линейных моделей. Подробнее данные метод мы разберем уже на четвертом шаге.

Дальше два кода, которые показывают как изменились наши данные после обработки выбросов для одномерных и многомерных методов соответственно.

In [ ]:
# Выбираем для примера один самый "выбросный" столбец (например, траты или возраст)
test_col = 'Fare'

# Объединяем статистики в одну красивую табличку для сравнения
compare_df = pd.DataFrame({
    'ДО обработки': X_train_replace[test_col].describe(),
    'ПОСЛЕ IQR': X_train_replace_iqr[test_col].describe()
})

print(compare_df)

In [ ]:
# 1. Проверяем изменение размера таблицы
print(f"Строк ДО очистки леса:    {X_train_replace_iqr.shape[0]}")
print(f"Строк ПОСЛЕ очистки леса: {X_train_replace_no_outliers.shape[0]}")
print(f"Удалено аномальных строк: {X_train_replace_iqr.shape[0] - X_train_replace_no_outliers.shape[0]}")
print("-" * 50)

# 2. Смотрим на сами выбросы, которые поймал Isolation Forest
# (берём строки, где метка была равна -1)
outliers = X_train_replace_iqr[train_labels == -1]

print("Пример аномальных пассажиров, которых нашел алгоритм:")
outliers.head(5) # Выведет топ-5 самых странных строк датасета

**Шаг третий.** Теперь вернемся к графикам из шага 1 и обработаем данные из проблемных гистограм. Этот шаг называется *трансформация формы распределения*. Этот шаг необходим для нормализации данных, чтобы распределения стало более похоже выглядеть на колокол и данные стали более сгруппированными.

Тут так же есть несколько различных вариантов трансформации, разберем их на наших проблемных столбцах. Все будем писать на примере работы первого алгоритва из шага два IQR.

**1.** Наверное, самый распространенный и удобное из всех способ - это логарифмирование. Можно использовать разныел логарифмы $log2,\ log10,\ log$. Но самый распространненый и частый для правого хвоста - это $log1p$. Используется чаще всего, если скос сильный skewness > 1.5, и в данных есть нули. Такой метод сжимает гигантские значения и растягивает мелкие. Это превратит скошенный график в более симметричный колокол.

In [ ]:
import numpy as np

X_train_log = X_train_replace_iqr.copy()
X_val_log = X_val_replace_iqr.copy()

# Находим оригинальные числовые столбцы
#num_cols = [col for col in X_train_log.select_dtypes(include='number').columns]

for col in num_cols:
    # Проверяем критерий скошенности строго по трейну
    if X_train_replace_iqr[col].skew() > 1.0:
        X_train_log[col] = np.log1p(X_train_log[col])
        X_val_log[col] = np.log1p(X_val_log[col])

**2.** Следующий частый способ - это степенные. Так же можно использовать много разных вариантов $sqrt,\ cbrt,\ x^2,\ x^3$. Обычно используют только квадратный корень. Так как корень сжимает данные мягче логарифма, то этот метод используется, если скос 1.0 < skewness < 1.5. Логарифм может оказаться слишком радикальным средством, которое перекосит данные уже в противоположную левую сторону.

In [ ]:
X_train_sqrt = X_train_replace_iqr.copy()
X_val_sqrt = X_val_replace_iqr.copy()

for col in num_cols:
    if X_train_replace_iqr[col].skew() > 1.0:
        X_train_sqrt[col] = np.sqrt(X_train_sqrt[col])
        X_val_sqrt[col] = np.sqrt(X_val_sqrt[col])

**3.** Третьим способом рассмотрим обратные преобразования, их так же много видов, но обычно используется или $1/x$, или $1/(x+1)$ - для избавления от нулей. 

Этот вариант сомнительный ведь он переварачивает маленькие значения и делает их огромными, а большие - наоборот. Способ идеально подходит, когда физический смысл признака логичнее инвертироватью Например, если есть *время на прохождение 1 км*, то для медленных машин этот признак превратится в *скорость*, распределение которой обычно гораздо более симметрично.

In [ ]:
X_train_inv = X_train_replace_iqr.copy()
X_val_inv = X_val_replace_iqr.copy()

for col in num_cols:
    if X_train_replace_iqr[col].skew() > 1.0:
        # Добавляем +1 для защиты от деления на ноль
        X_train_inv[col] = 1.0 / (X_train_inv[col] + 1.0)
        X_val_inv[col] = 1.0 / (X_val_inv[col] + 1.0)

**4.** Теперь переходим к более сложным трансформациям, которые одной функцией записать не получится. Рассмотрим сначала автоматическое преобразование Бокса-Кокса. Его все еще можно записать формулой: $$y = \begin{cases} \frac{x^\lambda - 1}{\lambda}, & \text{если } \lambda \neq 0 \\ \ln(x), & \text{если } \lambda = 0 \end{cases}$$

Алгоритм будет переебирать параметр $\lambda$ и искать такое его значение, которое сделает итоговое распределение максимольно близким к идеальному нормальному распределению. Плюс заключается в том что он может превратить данные в корень, логарифм, возвести в степень или инвертировать. **Важно!** Главное ограничение, что работает строго для чисел > 0.

In [ ]:
from sklearn.preprocessing import PowerTransformer

X_train_bc = X_train_replace_iqr.copy()
X_val_bc = X_val_replace_iqr.copy()

# Отбираем только те колонки, где skew > 1 и все значения строго положительные
bc_cols = [col for col in num_cols if X_train_replace_iqr[col].skew() > 1.0 and (X_train_replace_iqr[col] > 0).all()]

if bc_cols:
    pt_bc = PowerTransformer(method='box-cox')
    # Обучаемpt строго на трейне, трансформируем оба!
    X_train_bc[bc_cols] = pt_bc.fit_transform(X_train_bc[bc_cols])
    X_val_bc[bc_cols] = pt_bc.transform(X_val_bc[bc_cols])

**5.** Следуюзим рассмротим автоматическое преобразование Йео-Джонсона. Это модифицированная версия предыдущего метода Бокса-Кокса, созданное для того, чтобы убрать главное ограничение. Поэтому математическая формула очень сложная, но смысл работы остается такой же: подобрать параметр $\lambda$ для выравнивания распределения. Это универсальный и умный степенной скейлер, который сам подбирает функцию вырванивания и умеет работать с нулями и минусами. Поэтому его чаще выбирают вместо алгоритма Бокса-Кокса.

In [ ]:
from sklearn.preprocessing import PowerTransformer

X_train_yj = X_train_replace_iqr.copy()
X_val_yj = X_val_replace_iqr.copy()

yj_cols = [col for col in num_cols if X_train_replace_iqr[col].skew() > 1.0]

if yj_cols:
    pt_yj = PowerTransformer(method='yeo-johnson')
    X_train_yj[yj_cols] = pt_yj.fit_transform(X_train_yj[yj_cols])
    X_val_yj[yj_cols] = pt_yj.transform(X_val_yj[yj_cols])

**6.** Последним рассмотрим самый мощный метод квантильное выравнивание (Quantile Transformation). Он полностью игнорирует изначальную математическую формулу признака. Он берет ранги данных и принудительно разазывает их по теоретическим квантилям нормального распределения. Благодаря этому оно способно превратить распределение любого вида в идеальную, прямую линию на Q-Q Plot и безупречный колокол нормального распределения.

За такие идеальные возможности приходится платить. Из-за сврех-идеального распределения размывается пропорции и относительные расстрояние. Например, пассажир А потратил 10 и пассажир Б потратил 10 000. После логарифма расстояния уменьшатся, но модель всё равно будет четко понимать: Б потратил намного больше, чем А.

QuantileTransformer же переводит данные в ранги (порядковые номера). Ему важен только порядок объектов. Если между Б и А в датасете никто больше ничего не тратил, то Б и В на распределении окажутся совсем рядом. Абсолютная разница в деньгах (9 990) просто сотрется. Модель потеряет информацию о масштабе экономического превосходства клиента Б.

In [ ]:
from sklearn.preprocessing import QuantileTransformer

X_train_qt = X_train_replace_iqr.copy()
X_val_qt = X_val_replace_iqr.copy()

qt_cols = [col for col in num_cols if X_train_replace_iqr[col].skew() > 1.0]

if qt_cols:
    n_quant = min(500, X_train_replace_iqr.shape[0])

    # output_distribution='normal' принудительно заставляет ранги лечь на нормальную прямую
    qt = QuantileTransformer(n_quantiles=n_quant, output_distribution='normal', random_state=42)
    X_train_qt[qt_cols] = qt.fit_transform(X_train_qt[qt_cols])
    X_val_qt[qt_cols] = qt.transform(X_val_qt[qt_cols])
    

Результат посмотрим на прмиере $log1p$. Можно поменять на любую из таблиц выше.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target_col = 'Fare'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(X_train_replace_iqr[target_col], kde=True, ax=axes[0], color='coral')
axes[0].set_title(f'ДО (Skewness: {X_train_replace_iqr[target_col].skew():.2f})')

sns.histplot(X_train_log[target_col], kde=True, ax=axes[1], color='limegreen')
axes[1].set_title(f'ПОСЛЕ LOG1P (Skewness: {X_train_log[target_col].skew():.2f})')
plt.show()

Стало заметно лучше и график стал более похож на колокол.

**Шаг четвертый.** Масштрабирование признаков. Этот этап технически необходим большинству моделей. Без масштабирования признаки с большим масштабом будет полностью доминировать над признаком с маленьких масштабом.

Разберем это на примере: Если первый числовой столбец принимает значения от 0 до 10, а второй имеет минимальное зачение больше 1 000 000, то второй столбец надо будет преобразовывать. Ведь в линейных моделях веса первого столбца будут во много раз больше, чем веса второго столбца. Это создаст ненужный шум, который ухудшит точность нашей модели. Для нелинейных моделей, хоть такие данные и меньше влияют на точность, но все равно создают шум, который можно спокойно устранить при желании.

Рассмотрим самые распространненые типы масштабирования и напишем код на пример $log1p$.

**Первый тип.** Стандартизация (StandardScaler). Математическая формула:$$z = \frac{x - \mu}{\sigma}$$где $\mu$ - среднее арифметическое, а $\sigma$ - стандартное отклонение. 

Этот алгоритм сдвигает распределение так, что среднее значение признака становится равным 0, а диспресия и стандартное отклонение - 1. К сожалению, крайне чувствителен к выбросам. Если в данных остался огромный пик, он раздует $\sigma$, и все нормальные значения сожмутся в очень маленький интервал около нуля.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_train_scaled_st = X_train_log.copy()
X_val_scaled_st = X_val_log.copy()


scaler_st = StandardScaler()
# fit_transform делаем строго на ТРЕЙНЕ, на валидации только transform!
X_train_scaled_st[num_cols] = scaler_st.fit_transform(X_train_scaled_st[num_cols])
X_val_scaled_st[num_cols] = scaler_st.transform(X_val_scaled_st[num_cols])

**Второй тип.** Мин-Макс нормализация (MinMaxScaler). Математическая формула:$$x_{\text{scaled}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$

Алгоритм жестко сжимает все данные в фиксированный диапазон, от -1 до 1. Аналогично предыдущему алгоритму очень чувствителен к выбросам, поэтому обязательно необходимо что-то с ними сделать.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X_train_scaled_mm = X_train_log.copy()
X_val_scaled_mm = X_val_log.copy()

scaler_mm = MinMaxScaler()
X_train_scaled_mm[num_cols] = scaler_mm.fit_transform(X_train_scaled_mm[num_cols])
X_val_scaled_mm[num_cols] = scaler_mm.transform(X_val_scaled_mm[num_cols])

**Трерий тип.** RobustScaler. Этот способ, о котором было написано в шаге 2. Математическая формула: $$x_{\text{scaled}} = \frac{x - \text{median}}{Q_3 - Q_1}$$где $Q_1$ и $Q_3$ - это 25-й и 75-й перцентили, а их разница ($Q_3 - Q_1$) - это уже знакомый нам межквартильный размах $IQR$. 

Ключевой плюс в том что он лечит выбросы, вместо средних значений он использует медиану (центр), а вместо стандартного отклонения - IQR (ширину основной массы данных).  Поскольку перцентили не смотр на экстремальные значения, масштаб рассчитывается только по нормальной массе данных. Поэтому этот метод обычно используется когда на шаге 2 выбросы решили умышленно не обрезать через .clip()

In [ ]:
from sklearn.preprocessing import RobustScaler

# Берем данные ДО обрезки выбросов (например, сразу после заполнения пропусков с индикаторами)
X_train_scaled_rob = X_train_add_ind.copy() 
X_val_scaled_rob = X_val_add_ind.copy()

scaler_rob = RobustScaler()
X_train_scaled_rob[num_cols] = scaler_rob.fit_transform(X_train_scaled_rob[num_cols])
X_val_scaled_rob[num_cols] = scaler_rob.transform(X_val_scaled_rob[num_cols])

Есть еще два способа, но они были подробно расписаны на предыдущем шаге. Напишем только то, как они связаны с масштабированием: 
* **QuantileTransformer:** Этот алгоритм переводит данные в ранги и принудительно раскладывает по нормальному распределению. Поэтому этот алгоритм одновременно решает задачу и зименения формы, и изменения масштаба.
* **PowerTransformer (Yeo-Johnson / Box-Cox):** Эти два алгоритма находят оптимальную степень для выпрямления распределения, а после этого автоматически применяет стандартизацию (StandardScaler), приводя среднее к 0 и дисперсию к 1.


Поэтому тут не будем зацикливаться на этих алгоритмах и покажем просто то, как нашу таблицу изменили алгоритмы масштабирования. Написано на примере StandardScaler.

In [ ]:
import pandas as pd

test_col = 'Fare' 

# Собираем описательные статистики в одну таблицу для сравнения
compare_scaling = pd.DataFrame({
    'ДО масштабирования': X_train_log[test_col].describe(),    # Таблица из Шага 3
    'ПОСЛЕ StandardScaler': X_train_scaled_st[test_col].describe() # Итоговая таблица из Шага 4
})

# Настраиваем отображение чисел
pd.set_option('display.float_format', lambda x: '%.6f' % x)

print("Сравнение статистик для столбца:", test_col)
print(compare_scaling)

Ключевое что надо заметить - это изменение mean и std. Они стали 0  и 1 соответсвенно.

**Шаг пятый.** Теперь необходимо избавиться от мультиколлинеарности. Если два или более признаков очень сильно связаны друг с другом, то веса этих признаков при обучении модели улетают в бесконечность и знаки меняются на случайные, поэтому для линейных моделей мультиколлинеарные признаки смертельны. Для деревьев и бустрингов - это практически безвредно, поэтому для таких моделей пункт можно пропускать.

Рассмотрим **первый способ** матрицу корреляций (Пирсона). коэффициент корреляции Пирсона $r_{xy}$ между каждой парой числовых признаков. Он меняется от $-1$ до 1. На практике, если $|r_{xy}| > 0.8$ или $> 0.9$, один из этих двух признаков скорее всего надо удалять.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Строим матрицу корреляций Пирсона
corr_matrix = X_train_scaled_st[num_cols].corr()

# 2. Красиво её визуализируем через тепловую карту (Heatmap)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Матрица корреляции числовых признаков')
plt.show()

# 3. Автоматический поиск и составление списка сильно связанных признаков (порог > 0.8)
threshold = 0.8
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for col in upper_tri.columns if any(upper_tri[col].abs() > threshold)]

print(f"Столбцы, которые рекомендуется удалить из-за высокой корреляции: {to_drop}")

**Метод два.** Рассмотрим другой способ - фактор инфляции дисперсии (Variance Inflation Factor VIF). Матрица Пирсона ловит только парную корреляцию, поэтому VIF решает проблему, если признак сильно коррелирует с суммой других признаков, но недостаточно сильно с ними поотдельности. Формула VIF:$$VIF_i = \frac{1}{1 - R_i^2}$$где $R_i^2$ - коэффициент детерминации этой внутренней модели.

Если VIF = 1 корреляции нет вообще, если >5 то есть умеренная мультиколлинеарность, а если VIF > 10, то столбец очень сильно коллинеарен с другими. И, скорее всего, его надо будет удалять.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Создаем функцию для расчета VIF по таблице
def calculate_vif(df_numeric):
    vif_data = pd.DataFrame()
    vif_data["Признак"] = df_numeric.columns
    # Считаем VIF для каждого столбца i
    vif_data["VIF"] = [variance_inflation_factor(df_numeric.values, i) for i in range(df_numeric.shape[1])]
    return vif_data.sort_values(by="VIF", ascending=False)

# Запускаем расчет по нашим отскейленным оригинальным числам
vif_df = calculate_vif(X_train_scaled_st[num_cols])
print(vif_df)

Тут мы можем сделать вывод, что никакие столбцы между собой не коллинеарны настолько, чтобы дублировать друг друга. Поэтому удалть ничего мы не будем, но если бы удалять надо было бы, то код есть ниже

In [ ]:
# Финальный дроп после анализа
features_to_remove = [] 

X_train_vif_clean = X_train_scaled_st.drop(columns=features_to_remove)
X_val_vif_clean = X_val_scaled_st.drop(columns=features_to_remove)

Мы плавно переходим к следующему шагу. Мы поняли и удалили признаки, которые дублируют друг друга. Но что если осталось еще очень много признаков? С помощью алгоритмов эти признаки можно спокойно переработать в меньшее количество признаков. Поэтому

**Шестой шаг.** Снижение размерности. Этот шаг необходим по трем причинам: сжать слишком широкую таблицу, избавиться от размерности перед обучением моделей или спроецировать 10-20 признаков на плоскость, чтобы глазами увидеть кластеры и структуру данных. Рассмотрим каждый из видов снижения поотдельности


**Первый тип.** Метод главных компонент (Principal Component Analysis PCA). Этот алгоритм находит новые ортогональные оси, вдоль которых дисперсия данных максимальна. Алгоритм строит матрицу ковариации, находит ее собственные векторы, которые задают направления новых осей, и собственные значения, которые показывают долю сохраненной дисперсии. 

Этот алгоритм идеален для линейных моделей, он позволяет сжать 100 столбцов в 3 компоненты, сохранив, например, 94% всей информации.

Но этот алгоритм полностью уничтожает физический смысл признаков. И из плюса вытекает закономерный минус - очень плохо работает, если зависимости в данных нелинейны.

In [ ]:
from sklearn.decomposition import PCA

X_train_pca = X_train_vif_clean.copy()
X_val_pca = X_val_vif_clean.copy()

current_num_features = [col for col in X_train_pca.select_dtypes(include='number').columns] #if not col.endswith('_isnan')]

# n_components=2 означает, что мы сжимаем все числовые столбцы в 2 главные компоненты
pca = PCA(n_components=2, random_state=42)

# Обучаем строго на трейне!
train_pca_components = pca.fit_transform(X_train_pca[current_num_features])
val_pca_components = pca.transform(X_val_pca[current_num_features])

# Посмотрим, какую долю информации (дисперсии) удержали эти 2 компоненты
print(f"Сохранено дисперсии: {pca.explained_variance_ratio_.sum():.4f}")

# Создаем из них новые столбцы
X_train_pca['PCA_1'] = train_pca_components[:, 0]
X_train_pca['PCA_2'] = train_pca_components[:, 1]
X_val_pca['PCA_1'] = val_pca_components[:, 0]
X_val_pca['PCA_2'] = val_pca_components[:, 1]

**Второй тип.** t-SNE (t-Distributed Stochastic Neighbor Embedding). Тут алгоритм пытается перенести многомерное облако точек на плоскость так, чтобы обьекты, которые были близкими соседями в исходном пространстве, остались близкими соседями и на графике.

Это лучший алгоритм для визуализации сложных скрытых структур. На графике сразу видно четкие кучки-кластеры. 

Но на результатах t-SNE нельзя обучать финальные модели, ведб метод не умеет трансформирвать новые обьекты через метод .transform(). У него есть только метод fit_transform(), то есть он пересчитывает всю картинку заново при изменении данных. Также этот метод очень медлительный.

In [ ]:
from sklearn.manifold import TSNE

# На трейне запускаем fit_transform. Валидацию сюда совать нельзя - t-SNE не умеет делать transform()!
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
train_tsne = tsne.fit_transform(X_train_vif_clean[current_num_features])

# Строим красивый график, раскрасив точки по реальному таргету (например, y_train)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=train_tsne[:, 0], y=train_tsne[:, 1], hue=y_train, palette='viridis', alpha=0.7)
plt.title('Визуализация данных через t-SNE')
plt.show()

**Третий тип.** Третим типом рассмотрим UMAP (Uniform Manifold Approximation and Projection). Данный алгоритм строит нечеткие сложные геометрические графы связей в многомерном пространстве и отображает их на низкоразмерное пространство.

Работает как t-SNE, но в разы быстрее и гограздо луже сохраняет не только локальную структуру (близких соседей), но и глобальную (расстояние между самими кластерами). В откличие от t-SNE у UMAP есть рабочий метод .transform(), поэтому его компоненты можно использовать как реальные признаки для обучения моделей.

In [ ]:
#!pip install umap-learn

import umap

X_train_umap = X_train_vif_clean.copy()
X_val_umap = X_val_vif_clean.copy()

umap_model = umap.UMAP(n_components=2, random_state=1)
train_umap = umap_model.fit_transform(X_train_umap[current_num_features])
val_umap = umap_model.transform(X_val_umap[current_num_features])

X_train_umap['UMAP_1'] = train_umap[:, 0]
X_train_umap['UMAP_2'] = train_umap[:, 1]
X_val_umap['UMAP_1'] = val_umap[:, 0]
X_val_umap['UMAP_2'] = val_umap[:, 1]

**Четвертый тип.** LDA (Lineral Discriminant Analysis). В отличии от PCA, который ищет просто максимальны разброс, LDA смотрит на целевые переменные y_train. Он находит такие оси, на которых проекции разных классов максимально удалены друго от друга, а разброс внутри каждого класса минимален.

Такой алгоритм идеально подготавливает признаки для классификаторов.

Однако максимальное количество компонент, которые может выдать LDA, строго ограничено формулой K-1, где K - количество классов в таргете. Для бинарной классификации LDA вернет ровно одну компоненту

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

X_train_lda = X_train_vif_clean.copy()
X_val_lda = X_val_vif_clean.copy()

# LDA требует на вход y_train, так как это метод с учителем!
lda = LinearDiscriminantAnalysis(n_components=1) 
train_lda = lda.fit_transform(X_train_lda[current_num_features], y_train)
val_lda = lda.transform(X_val_lda[current_num_features])

X_train_lda['LDA_feature'] = train_lda[:, 0]
X_val_lda['LDA_feature'] = val_lda[:, 0]

С изменением числа признаков мы разобрались. Если в шаге 5 мы убирали признаки, которые дублируют друг друга, то теперь настало время убирать признаки, которые бесполезны для предскаания таргета (Y).

**Шаг седьмой.** Отбор признаков. Рассмтрим несколько групп признаков:


**Первая группа.** Статистические фильтры (Filter Methods). Эти методы оценивают каждый признак незаисимо от модели машинного обучения. Они смотрят на чистую статистику связи между конкретной колонкой и таргетом. 

**1. f_classif (ANOVA F-valude).** Алгоритм проверяет гипотезу о равенстве средних значений непрерывного признака в группах, разделенных по классам таргета. Рассчитывает отношение мегрупповой дисперсии к внутригрупповой. Чем выше F-score, тем сильнее признак разделяет целевые классы. 

Плюс заключается в том, что работает мгновенно, идеально подходит как самый первый грубый фильтр для гигантских датасетов.

Но оценивает только линейную связь, полностью игнорируя нелинейне паттерны и парные взаимодействия признаков

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X_train_anova = X_train_vif_clean.copy()
X_val_anova = X_val_vif_clean.copy()
current_num_features = [col for col in X_train_anova.select_dtypes(include='number').columns ]#if not col.endswith('_isnan')]

# Отбираем 5 лучших фичей по тесту ANOVA
anova_selector = SelectKBest(score_func=f_classif, k=min(5, len(current_num_features)))
X_train_anova_arr = anova_selector.fit_transform(X_train_anova[current_num_features], y_train)
X_val_anova_arr = anova_selector.transform(X_val_anova[current_num_features])

anova_features = anova_selector.get_feature_names_out(current_num_features)
print(f"ANOVA оставил признаки: {list(anova_features)}")

**2. mutual_info_classif (Взаимная информация).** Нелинейный метод, основанный на энтропии Кульбака-Лейблера и оценке расстояний до k-ближайших соседей. Измеряет количество информации, которое можно получить о таргете, зная значение признака.

Самое главное преимущество - это способность отловить абсоллютно любые нелинейные, волновые, круговые и хаотические зависимости.

За функционал прийдется заплатить временем. Метод требует вычисления расстояний, поэтому работает ощутимо медленее, чем ANOVA. Причем на маленьких выборках склонен переобучаться под случайный шум.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

X_train_mi = X_train_vif_clean.copy()
X_val_mi = X_val_vif_clean.copy()

# k=5 фичей по критерию взаимной информации
mi_selector = SelectKBest(score_func=mutual_info_classif, k=min(5, len(current_num_features)))
X_train_mi_arr = mi_selector.fit_transform(X_train_mi[current_num_features], y_train)
X_val_mi_arr = mi_selector.transform(X_val_mi[current_num_features])

mi_features = mi_selector.get_feature_names_out(current_num_features)
print(f"Mutual Info оставил признаки: {list(mi_features)}")

**3. chi2 (Хи-квадрат).** Это статистический тест независимости признаков. Проверяет, есть ли значимые отклонения между наблюдаемыми и теоритическими (случайными) частотами совместного распределения признака и классов таргета.

Это отличный математический интсрумент для разреженных матриц и закодированных текстов.

Но математически неприменим к отрцательным числам. Если был применен на шаге 4 метод StandardScaler, то chi2 просто упает с ошибкой, так как стандартизация создает отрицательные значения.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

X_train_chi = X_train_vif_clean.copy()
X_val_chi = X_val_vif_clean.copy()

# Важно: chi2 требует неотрицательные значения. Временно вернем их в диапазон [0, 1] через MinMaxScaler
scaler_tmp = MinMaxScaler()
X_tr_non_neg = scaler_tmp.fit_transform(X_train_chi[current_num_features])
X_val_non_neg = scaler_tmp.transform(X_val_chi[current_num_features])

chi_selector = SelectKBest(score_func=chi2, k=min(5, len(current_num_features)))
X_train_chi_arr = chi_selector.fit_transform(X_tr_non_neg, y_train)

chi_features = chi_selector.get_feature_names_out(current_num_features)
print(f"Chi2 оставил признаки: {list(chi_features)}")

**Вторая группа.** Методы-обертки (Wrapper Methods). Эти методы используют реальную модель (например, логистическую регрессию) как черный ящик. Они перебирают разные подмножества признаков, обучают модель и смотрят, на каком наборе метрика качества (например, ROC_AUC) будет максимальной.

**4. Рекурсивное удаление признаков (RFE).** Жадный алгоритм сверху-вниз. Модель обучается на полном наборе. По значениям весоф находится наименее важный признак и физически вырезается из таблицы. Процесс циклически повторяется, пока не останется требуемое число фичей.

Этот метод учитывает особенности конкретной модели, выкидывает признаки, которые становятся бесполезными именно в присутсвии других столбцов.

Но требует многкратного переобучения тяжелых моделей на каждом шаге. Если признаков очень много, будет выполняться часами.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

X_train_rfe = X_train_vif_clean.copy()
X_val_rfe = X_val_vif_clean.copy()

base_lr = LogisticRegression(max_iter=1000, random_state=1)
rfe_selector = RFE(estimator=base_lr, n_features_to_select=min(4, len(current_num_features)))

rfe_selector.fit(X_train_rfe[current_num_features], y_train)
rfe_features = [col for col, keep in zip(current_num_features, rfe_selector.support_) if keep]
print(f"RFE оставил признаки: {rfe_features}")

**5. Жадный пошаговый отбор (SequentialFeatureSelector).** Алгоритм строит процесс перебора через кросс-валидацию. Режим direction='forward' стартует с пустой таблицы, перебирает признаки по одной, считает ошибку и ифксирует ту, которая дала максимальны прирост метрики. Процесс идет снизу-вверх.

Алгоритм находит математически оптимальные комбинации и синергию между признаками, не привязываясь к внутренним коэффицентам модели. 

Но это самый медленный алгоритм из всех существуюзих в ML.

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

X_train_sfs = X_train_vif_clean.copy()
X_val_sfs = X_val_vif_clean.copy()

base_lr2 = LogisticRegression(max_iter=1000, random_state=42)
sfs_selector = SequentialFeatureSelector(base_lr2, n_features_to_select=min(3, len(current_num_features)), 
                                         direction='forward', cv=3, n_jobs=-1)

sfs_selector.fit(X_train_sfs[current_num_features], y_train)
sfs_features = sfs_selector.get_feature_names_out(current_num_features)
print(f"SFS Forward оставил признаки: {list(sfs_features)}")

**Третья группа.** Встроенные методы (Emvedded Methods). Эти методы отбирают автоматически внутри самой модели в процессе ее обучения. Нам не нужно запускать сторонние модели.

**1. L1-регуляризация (Lasso).** Алгоритм добавляет к стандартной функции потерь штраф за сумму абсолютных значений весов ($L1 = \alpha \sum |w_i|$). Из-за геометрии ромбовидных ограничений L1-штрафа оптимизатор принудительно зануляет веса наименее полезных колонок, превращая их коэффициенты строго в 0.

Благодаря этому отрабатывает за один проход обучения модели, идеально подходит для отбора признаков перед запуском финальных линейных моделей.

Но накладывает сильное линейное ограничение. Если между признаком и таргетом сложная нелинейная связь, Lasso занулит этот признак, посчитав его бесполезным.

In [ ]:
from sklearn.linear_model import LogisticRegression

X_train_l1 = X_train_vif_clean.copy()

# penalty='l1' и solver='liblinear' активируют Lasso
lasso_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.2, random_state=1)
lasso_model.fit(X_train_l1[current_num_features], y_train)

# Вытаскиваем коэффициенты и смотрим, кто не равен нулю
lasso_coefs = pd.Series(lasso_model.coef_[0], index=current_num_features)
lasso_selected_features = lasso_coefs[lasso_coefs != 0].index.tolist()

print(f"L1-регуляризация оставила признаки (вес не 0): {lasso_selected_features}")

**2. Важность признаков в деревьях решений (feature_importances_).** Если L1-регуляризация предназначалась для линейных моделей, feature_imortances предназначен для деревьев и бустингов. Алгоритм обучает ансамбль, считает метрику MDI (Mean Decrease in Impurity) - насколько сильно расщепление по данному признаку снижает неопределенность/энтропию во всех узлах всех деревьев. Признаки с важностью ниже выбранного порога отбрасываются.

Не зависит от линейности данных, отлично ловит сложные нелинейные структуры и пороговые зависимости.

Но встроенный подсчет MDI в sklearn имеет серьезное математическое смещение - он искуственно завышае важность непрерывных признаков с большим количеством уникальных значений (высокой кардинальностью). 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

X_train_rf = X_train_vif_clean.copy()
X_val_rf = X_val_vif_clean.copy()

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# SelectFromModel автоматически отберет признаки со скором выше среднего (threshold='mean')
embed_rf_selector = SelectFromModel(estimator=rf, threshold='mean')
X_train_rf_arr = embed_rf_selector.fit_transform(X_train_rf[current_num_features], y_train)
X_val_rf_arr = embed_rf_selector.transform(X_val_rf[current_num_features])

rf_features = embed_rf_selector.get_feature_names_out(current_num_features)
print(f"Random Forest оставил признаки: {list(rf_features)}")

Возникает закономерный вопрос, а почему на шаге 7 мы использовали датасет из шага 5? Ответ на него очень прост: мы либо используем Шаг 6, либо исопльзуем Шаг 7. 

Шаг 6 используется, если
1. Признаков слишком много: больше 100 признаков.
2. Необходима визуализировать, показать распределение данных на плоскости (t-SNE, UMAP или PCA).
3. Признаки - это шум. Иногда исходные данные - это грязные измерения, которые по отдельности несут мало смысла, но их комбинация содержит четкий сигнал (PCA).

Шаг 7 же используется, если
1. Умеренное количество признаков: всего 20-50 признаков.
2. Критически важна интерпретируемость, где признаки должны что-то физически обозначать, а не столбец "компонента 3".
3. Используются линейные модели или деревья. Линейным моделям гораздо проще учиться на отобранных родных признаках.

**Шаг восьмой.** Перейдем к предпоследнему шагу по обработке численных данных - Бинаризации и Дискретизации.

Этот этап заключается в превращении непрерывных числовых признаков в интервальные или бинарные категории. Такой метод повышает качество линейных моделей, ведь такая модель предполагает, что с ростом признака таргет меняется монотонно. Но в жизни это может быть не так. Например, младенцы и пожилые люди имеют менее высокую вероятность выжить на Титанике, а молодые мужчины - высокую. И разница между возрастом 17 и 25 исчезает. Дискретизация режет возрасть на отрезки, позволяя каждому возрасту сопоставить свой индивидуальный вес. 
Так же небольшие изменения в стоимости билета 2 000 рублей и 2 100 рублей часто не несут в себе никакой логики. Обьединяя их в одну группу сглаживает данные.

Поняв, чем этот шаг важен, необходимо понять какими способами это достижимо. Всего существует 4 различных способа, разберем их подробно ниже:

**1. Разбиение на равные интервалы (pd.cut).** Суть этого метода заключается в том, что берется диапазон от минимального до максимально значения и делится на N абсолютно равных по ширине кусков.

Такой метод легко интерпреировать, например, группы: 0-20 лет, 20-40 лет, 40-60 лет.

Но этот алгоритм крайне чувствителен к пустотам в данных. Если есть 100 пассажиров с билетами по 2 000 рублей и один с билетом 10 000, то pd.cut разобьет на 5 интервалов создаст корзины, где в первой будут сидеть все 100 человек, а в остальных - никого.

In [ ]:
X_train_bin1 = X_train_vif_clean.copy()
X_val_bin1 = X_val_vif_clean.copy()

# Нарезаем возраст на 4 равных по ширине отрезка. 
# labels=False возвращает номера корзин (0, 1, 2, 3) вместо текстовых интервалов, что нужно моделям
X_train_bin1['Age_bins'] = pd.cut(X_train_bin1['Age'], bins=4, labels=False)

# Чтобы не было утечки, валидацию нужно резать по границам, которые сформировались на ТРЕЙНЕ!
# Для этого вытаскиваем границы через категориальное свойство (bins)
_, train_bins = pd.cut(X_train_bin1['Age'], bins=4, retbins=True)
X_val_bin1['Age_bins'] = pd.cut(X_val_bin1['Age'], bins=train_bins, labels=False, include_lowest=True)

**2. Квантильное разбиение (pd.qcut).** На помощь первому способу приходит второй. Этот метод делить данные так, чтобы в каждый интервал попало одинаковое количество строк. Границами групп становятся квантили распределения.

Этот метод идеально распределяет наполненность корзин. Он защищен от проблемы пустых интервалов при скошенных распределениях.

Но границы у такого метода получаются математически некрасивыми (нецелыми).



In [ ]:
X_train_bin2 = X_train_vif_clean.copy()
X_val_bin2 = X_val_vif_clean.copy()

# Нарезаем стоимость билета так, чтобы в каждой корзине было одинаковое количество людей
X_train_bin2['Fare_bins'], train_q_bins = pd.qcut(X_train_bin2['Fare'], q=4, labels=False, retbins=True)

# Применяем зафиксированные границы квантилей к валидации
X_val_bin2['Fare_bins'] = pd.cut(X_val_bin2['Fare'], bins=train_q_bins, labels=False, include_lowest=True)

**3. Автоматический трансформер (KBinsDiscretizer).** Это профессиональный инструмент из scikit-learn. Он умеет автоматически разбивать данные по разным стратегиям (uniform - аналог pd.cut, quantile - аналог pd.qcut, kmeans - кластеризация данных на группы).

Самый главный плюс, что этот метод работает как полноценный склейлер (есть метод .transform()), что позволяет обучить границы на train и применить их к валидации без утечки данных. Интервалы можно сразу кодировать в OHE-матрицу.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

X_train_kbd = X_train_vif_clean.copy()
X_val_kbd = X_val_vif_clean.copy()

# encode='ordinal' выдает номера групп. strategy='quantile' распределяет строки поровну.
kbd = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')

# Обучаем строго на трейне по выбранным столбцам
X_train_kbd[['Age_kbd', 'Fare_kbd']] = kbd.fit_transform(X_train_kbd[['Age', 'Fare']])
X_val_kbd[['Age_kbd', 'Fare_kbd']] = kbd.transform(X_val_kbd[['Age', 'Fare']])

**4. Жетская бинаризация (Binarizer).** Этот метод превращает число в 0 или 1 по фиксированному порогу.

Особенных плюсов и минусов выделить нельзя, но на примере билетов на Титанике, метод мождно применить к стоимости билета, если Fare > 0, то пассажир купил билет (1), если Fare = 0, то этот человек - член экипажа.

In [ ]:
from sklearn.preprocessing import Binarizer

X_train_binarized = X_train_vif_clean.copy()
X_val_binarized = X_val_vif_clean.copy()

# Устанавливаем порог threshold=0.0. Всё что выше нуля станет 1, всё что 0 или ниже - 0.
binarizer = Binarizer(threshold=0.0)

X_train_binarized['Is_Paid_Passenger'] = binarizer.fit_transform(X_train_binarized[['Fare']])
X_val_binarized['Is_Paid_Passenger'] = binarizer.transform(X_val_binarized[['Fare']])

**Шаг девятый.** Генерация новых признаков (Feature Engineering).

Самый творческий шаг из всех, ведь нету определенной модели, по которой создаются новые признаки. Каждый может сам выбрать как он хочет переплетать признаки. Этот шаг используется не только для численных данных. Его можно использовать как для категориальных, так и для текстовых столбцов. Но больше всего разнообразия выбора возникает именно с численными столбцами. Существует несколько методов, вариантов, генерации признаков. Разберем их:

**1. Полиномиальные признаки (PolynomialFeatures).** Суть этого метода заключается в создании новых столбцом путем возведения существующих признаков в степень, а также премножения их между собой. 

Такой метод позволяет линейным моделям находить сложные, нелинейные (параболические, круговые) границы разделения классов. Обычная логистиеская регрессия на полиномах начинает работать с точностью сложных нелинейных моделей.

С другой стороны, росто размерности таблицы. Ведь создается очень большое количество новых столбцов. Если неа вход подать 10 столбцов, полином 3-й степени создаст сотни новых столбцов. Модель начнет переобучаться и требовать коллосального количества оперативной памяти.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_train_poly = X_train_vif_clean.copy()
X_val_poly = X_val_vif_clean.copy()

# Нам нужны только базовые непрерывные фичи (например, Age и Fare)
target_num_cols = ['Age', 'Fare']

# include_bias=False убирает единичный столбец (интерцепт)
poly = PolynomialFeatures(degree=2, include_bias=False)

# Обучаем полиномы строго на ТРЕЙНЕ, трансформируем оба датасета
train_poly_features = poly.fit_transform(X_train_poly[target_num_cols])
val_poly_features = poly.transform(X_val_poly[target_num_cols])

# Получаем красивые автоматические имена столбцов (типа "Age^2", "Age Fare")
poly_cols = poly.get_feature_names_out(target_num_cols)

# Превращаем массивы обратно в DataFrame и склеиваем с исходными таблицами
train_poly_df = pd.DataFrame(train_poly_features, columns=poly_cols, index=X_train_poly.index)
val_poly_df = pd.DataFrame(val_poly_features, columns=poly_cols, index=X_val_poly.index)

# Добавляем новые колонки (исключая исходные Age и Fare, чтобы они не дублировались)
new_poly_cols = [c for c in poly_cols if c not in target_num_cols]
X_train_poly[new_poly_cols] = train_poly_df[new_poly_cols]
X_val_poly[new_poly_cols] = val_poly_df[new_poly_cols]

**2. Парные взаимодействия (Ручные математические операции).** Самый простой метод - проектирование признаков на основе логики и здравого смысла с помощью базовой арифметики. Например, в нашей таблице у нас есть очень много разных трат. Сложив их мы получим признак - всего потрачено денег. Поделив это на возраст, мы получим интенсивность трат на год жизни.

Этот метод сохраняет идеальный физический и понятный человеку смысл. Модель становится в разы сильнее без раздувания таблицы.

Очевидный минус, что метод требует ручного разведочного анализа данных и глубокого погружения в логику.

In [ ]:
X_train_eng = X_train_vif_clean.copy()
X_val_eng = X_val_vif_clean.copy()

# Предположим, у нас Spaceship Titanic с тратами: RoomService, FoodCourt, ShoppingMall, Spa, VRDeck
spend_cols = [c for c in ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'] if c in X_train_eng.columns]

if spend_cols:
    # 1. Суммарные траты
    X_train_eng['Total_Spend'] = X_train_eng[spend_cols].sum(axis=1)
    X_val_eng['Total_Spend'] = X_val_eng[spend_cols].sum(axis=1)
    
    # 2. Логический флаг: тратил ли человек деньги вообще (1 - тратил, 0 - экономил)
    X_train_eng['Is_Spender'] = (X_train_eng['Total_Spend'] > 0).astype(int)
    X_val_eng['Is_Spender'] = (X_val_eng['Total_Spend'] > 0).astype(int)

# 3. Относительный признак: Отношение стоимости билета к возрасту (с защитой от деления на 0)
X_train_eng['Fare_per_Age'] = X_train_eng['Fare'] / (X_train_eng['Age'] + 1.0)
X_val_eng['Fare_per_Age'] = X_val_eng['Fare'] / (X_val_eng['Age'] + 1.0)

**3. Групповые статистики и агрегации (groupby + transform).** Этот метод представляет собой расчет статистических показателей для числового признака, сгруппированных по какому-то категориальному полю. 

Пример: узнать, какая медианная стоимость билета была у пассажиров конкретно из 1-го класса, и записать это значение каждому человеку.

Этот метод позволяет модели сравнивать текущего пассажира с его социальной группой. 
Но очень большая вероятность утечки данных. Ведь статистику нужно считать строго по тренировочной выборке, а затем просто переносить эти константы на валидацию. Иначе мы закинем в таблицу данные, по которым мы дальше проверяем точность нашей модели.

In [ ]:
X_train_agg = X_train_vif_clean.copy()
X_val_agg = X_val_vif_clean.copy()

# Группируем по категориальному признаку (например, 'Pclass' - класс каюты) 
# и считаем среднее для числового признака (например, 'Fare')
group_col = 'Pclass'
target_calc_col = 'Fare'

if group_col in X_train_agg.columns:
    # ШАГ 1: Считаем карту средних значений СТРОГО по тренировочному датасету
    mean_mapping = X_train_agg.groupby(group_col)[target_calc_col].mean()
    
    # ШАГ 2: Мэппим полученные константы и в ТРЕЙН, и в ВАЛИДАЦИЮ
    X_train_agg['Pclass_Mean_Fare'] = X_train_agg[group_col].map(mean_mapping)
    X_val_agg['Pclass_Mean_Fare'] = X_val_agg[group_col].map(mean_mapping)
    
    # На случай, если в валидации встретится редкая новая категория, которой не было в трейне
    # заполняем образовавшийся пропуск общим средним по трейну
    X_val_agg['Pclass_Mean_Fare'] = X_val_agg['Pclass_Mean_Fare'].fillna(X_train_agg[target_calc_col].mean())

С численными столбцами все! Теперь надо обработать два оставшихся типа: категориальные и текстовые столбцы. Начнем, пожалуй, с категориальных и рассмотрим их поподробнее. 

Выделим 6 шагов работы с категориальными признаками:
1. Анализ
2. Взаимодействия между категориальными признаками (Feature Engineering)
3. Обработка редких категорий
4. Encoding - основное
5. Обработка новых категорий
6. Снижение размерности после OHE

**Шаг первый.** Начнем с анализа категориальных признаков и статистики. Перед тем как модифицировать данные, нужно получить полную картину. Для этого пройдемся по четырем пунктам:
1. **Количество уникальных значений (Cardinality).** Этот параметр покажет, сколько у нас всего есть различных значений. Для малого значения идеальным будет One-Hot Encoding, а для высокого значения идеально подойдет Target, Frequency или Hashing Encoding. Все это будет разобрано далее.
2. **Частота каждого значения (value_couts).** Этот параметр показывает распределение обьектов по категориям. Он помогает увидлеть, есть ли доминирующие классы или гигнатские хвосты из очень редких значени, которые на следующем шаге нудно будет схлопывать в общую корзину.
3. **Связь с таргетом.** Показывает долю положительного класса для каждой категории. Если у категории "A" доля 0.9, а у "B" - 0.1, этот признак имеет огромную предсказательную силу для будущих моделей.
4. **Есть ли порядок в категориях (Ordinal или Nominal).**
   * Порядковые категории (Ordinal) имеют внутреннюю шкалу. Их нужно будет кодировать как последовательные числа
   * Номинальные категории (Nominal) равноправны, между ними нельзя поставить знак больше или меньше. Их порядок кодировать числами напрямую нельзя - это запутает линейные модели.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Автоматически находим все категориальные/текстовые/булевые столбцы в датасете
cat_cols = X_train_replace.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f"Категориальные признаки: {cat_cols} \n")

for col in cat_cols:
    print(f"==================================================")
    print(f"Анализ признака: '{col}'")
    print(f"==================================================")
    
    # 1. Считаем кардинальность (количество уникальных)
    cardinality = X_train_replace[col].nunique(dropna=True)
    print(f"• Кардинальность (число уникальных значений): {cardinality}")
        
    # 2. Частотный анализ (value_counts) - выводим Топ-10 значений
    print("\n• Частота каждого значения (Топ-10):")
    counts = X_train_replace[col].value_counts(dropna=False)
    for val, count in counts.head(10).items():
        pct = (count / len(X_train_replace)) * 100
        print(f"  - Категория '{val}': {count} строк ({pct:.2f}%)")
        
    # 3. Связь с бинарным таргетом
    # Временно объединяем для группировки
    temp_df = pd.DataFrame({col: X_train_replace[col], 'target': y_train})
    
    # Считаем среднее значение таргета на категорию (долю 1)
    target_mapping = temp_df.groupby(col)['target'].mean().sort_values(ascending=False)
    
    print("\n• Связь с таргетом (доля положительного класса):")
    for val, mean_target in target_mapping.head(10).items():
        print(f"  - При значении '{val}' -> Средний таргет = {mean_target:.4f}")
            
        # 4. Визуализация связи с таргетом (строим только для тех, у кого не слишком много значений)
        if cardinality <= 15:
            plt.figure(figsize=(10, 4))
            sns.barplot(x=temp_df[col].astype(str), y=temp_df['target'], palette='coolwarm', errorbar=None)
            plt.title(f"Зависимость таргета от признака '{col}'")
            plt.ylabel("Доля положительного класса")
            plt.grid(axis='y', linestyle='--', alpha=0.5)
            plt.xticks(rotation=45)
            plt.show()

**Шаг второй.** Взаимодействие между категориальными признаками (Feature Engineering). Очень подробно было расписано в шаге 9 при обработке численных данных, поэтому тут зацикливаться над теорией нет смылса. Распишем просто кодом два варианта

**1. Рунчая строковая конкатенация через Pandas** Самый частый способ. 

Есть полный контроль над именами признаков, на выходе получается чистый, понятный текст. Но нужно прописывать кажду. пару вручную.

In [ ]:
X_train_inter = X_train_replace.copy()
X_val_inter = X_val_replace.copy()

# Задаем пары для склеивания (например, Ticket и Cabin)
col1, col2 = 'Ticket', 'Cabin'

# Склеиваем столбцы через нижнее подчеркивание. 
# astype(str) гарантирует защиту от ошибок, если тип данных изначально не object
X_train_inter[f'{col1}_{col2}'] = X_train_inter[col1].astype(str) + '_' + X_train_inter[col2].astype(str)
X_val_inter[f'{col1}_{col2}'] = X_val_inter[col1].astype(str) + '_' + X_val_inter[col2].astype(str)

print(f"Успешно создан комбинированный признак: {col1}_{col2}")
print(X_train_inter[[f'{col1}_{col2}']].head(3))

**2. Автоматическое парное кодирование всех категорий.** Автоматически перебирает указанные колонки и собирает из них уникальные строки. Но название столбцов будут генерироватся как безликие индексы

In [ ]:
# Если мы хотим скомбинировать сразу несколько полей, можно использовать zip в цикле
X_train_auto_inter = X_train_replace.copy()
X_val_auto_inter = X_val_replace.copy()

target_pair_cols = ['Embarked', 'Ticket', 'Cabin']

# Генерируем парные взаимодействия между всеми выбранными колонками
for i in range(len(target_pair_cols)):
    for j in range(i + 1, len(target_pair_cols)):
        c1, c2 = target_pair_cols[i], target_pair_cols[j]
        new_name = f"{c1}_x_{c2}"
        
        X_train_auto_inter[new_name] = X_train_auto_inter[c1].astype(str) + "_" + X_train_auto_inter[c2].astype(str)
        X_val_auto_inter[new_name] = X_val_auto_inter[c1].astype(str) + "_" + X_val_auto_inter[c2].astype(str)

print("Автоматически сгенерированные взаимодействия категорий:")
print([col for col in X_train_auto_inter.columns if "_x_" in col])

**Шаг третий.** Обработка редких категорий. Этот шаг аналогичен шагу с обработкой выбросов для числовых значений, поэтому его очень важно выполнить перед обучением модели.

**Способ первый.** Группировка в "Other" по минимальному порогу частоты. Это самый популярный подход.

Мы задаем пороговое значение. Оно может быть абсолютно любым: константным (хотя бы 10 раз) или относительным (хотя бы 1% от всего датасета). Все уникальные значения которые не дотянули до этого порога, насильно переименовываются в "Other" или "Rare".

Очень сильно падает кардинальность признака. Модель защищена от переобучения на редких аномалиях и OHE-матрица получается компактной.

Если в категорию "Other" сливается слишком много принципиально разных обьектов, модель может потерять предсказательный сигнал, который был размазан по этим редким классам.

In [ ]:
X_train_rare1 = X_train_inter.copy()
X_val_rare1 = X_val_inter.copy()

# Выбираем столбцы для обработки
cols_to_clean = ['Ticket_Cabin', 'Cabin']
threshold_pct = 0.01 # Порог 1%

for col in cols_to_clean:
    # 1. Считаем относительные частоты строго по ТРЕЙНУ
    freqs = X_train_rare1[col].value_counts(normalize=True)
    
    # 2. Фильтруем те категории, которые преодолели порог в 1%
    frequent_categories = freqs[freqs >= threshold_pct].index.tolist()
    
    # 3. Всё, что не вошло в список частых, заменяем на 'Other' в обоих датасетах
    X_train_rare1[col] = X_train_rare1[col].apply(lambda x: x if val in frequent_categories else 'Other')
    X_val_rare1[col] = X_val_rare1[col].apply(lambda x: x if val in frequent_categories else 'Other')

print("Пример распределения после относительного сжатия:")
print(X_train_rare1['Ticket_Cabin'].value_counts())


X_train_rare2 = X_train_inter.copy()
X_val_rare2 = X_val_inter.copy()

min_rows_threshold = 10 # Категория должна встретиться минимум 10 раз

for col in cols_to_clean:
    # 1. Считаем абсолютное количество строк по ТРЕЙНУ
    counts = X_train_rare2[col].value_counts()
    
    # 2. Оставляем только те категории, у которых строк >= 10
    frequent_categories = counts[counts >= min_rows_threshold].index.tolist()
    
    # 3. Схлопываем хвост в 'Other'
    X_train_rare2[col] = X_train_rare2[col].apply(lambda x: x if val in frequent_categories else 'Other')
    X_val_rare2[col] = X_val_rare2[col].apply(lambda x: x if val in frequent_categories else 'Other')

print("Пример распределения после абсолютного сжатия:")
print(X_train_rare2['Cabin'].value_counts().head(5))

**Способ второй.** Удалить строки с редкими категориями. Необходимо найти строки, где содержаться экстремально редкие значения, и полностью выбросить их из таблицы через .drop()

Датасет очищается от явного мусора и опечаток.

Но этот метод можно применять строго только к тренировочной выборке. Если редкая категория придет на валидации или тесте, то нельзя ее удалять. Поэтому на практике строки удаляют крайне редко, используя первый способ

In [ ]:
X_train_rare3 = X_train_inter.copy()

# Допустим, мы хотим полностью стереть из ТРЕЙНА строки, если категория встретилась всего 1 раз
for col in cols_to_clean:
    counts = X_train_rare3[col].value_counts()
    single_time_categories = counts[counts == 1].index.tolist()
    
    # Физически выкидываем эти строки из обучения
    X_train_rare3 = X_train_rare3[~X_train_rare3[col].isin(single_time_categories)] # ~ означает логическое не

print(f"Размер тренировочной выборки после удаления ультра-редких строк: {X_train_rare3.shape}")

**Шаг четвертый.** Encoding (Кодирование катеогориальных признаков). Основной и самый емкий шаг, которые необходимо выполнить при работе с категориальными признаками.

**1. LabelEncoder.** Самый первый и простой вариант - это перевоить текстовые значения категорий в последовательные числа 0,1,2... в соответсвии с заданными строгим алфавитным порядком. 

Работает мгновенно, не раздувает размерности матрицы. Хорошо переваривается нелинейными моделями.

Но принудительно вводит ложный математический порядок там, где его нет. По факту этот класс спроектирован исключительно для кодирования нечисловой целевой переменной. Использовать его для кодирования признраков неуместно, так как он одномерный и не умеет кодировать несколько столбцов за раз. У него нет параметра для защиты от новых категорий: если на валидации или тесте модель встретит слово, которого не было в обучении, код упадет с ошибкой.

**2. OrdinalEncoder.** На помощь первому методу приходит этот. Он переводит текстовые значения категорий в последовательные числа 0,1,2... в соответсвии с заданными строгим логическим порядком. Этот порядок выбирается произвольно, поэтому необходимо очень четко понимать для чего нужен этот метод и как разбивать данные.

Очевидно, этот метод подходит для категорий, где физический порядок имеет значение (например, класс обслуживания: Эконом $\to$ Бизнес $\to$ Первый). Такой метод не раздувает таблицу, ведь остается всего один числовой столбец.

Но если применить алгоритм к равноправным признакам, он принудительно введет математический порядок, что очень пагубно скажется на точности линейных моделей.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X_train_ord = X_train_rare1.copy() # отталкиваемся от данных после Шага 3
X_val_ord = X_val_rare1.copy()

# Явно задаем иерархию категорий (от меньшего к большему)
# Предположим, кодируем какой-то признак уровня VIP или класса обслуживания
pclass_order = [['Third', 'Second', 'First']] 

# use_encoded_value + unknown_value=-1 настраивают стратегию unknown для новых значений
ord_enc = OrdinalEncoder(categories=pclass_order, handle_unknown='use_encoded_value', unknown_value=-1)

X_train_ord['Pclass_encoded'] = ord_enc.fit_transform(X_train_ord[['Pclass']])
X_val_ord['Pclass_encoded'] = ord_enc.transform(X_val_ord[['Pclass']])

**3. OneHotEncoder (OHE).** Один из самых простых и популярных методов. Создает для каждого уникального значения категории свой собственный бинарный столбец.

Математически метод прозрачен и не вводит ложного порядка, сохраняя равные расстояния между обьектами. Стандарт по умолчанию для линейных моделей.

Но, к сожалению, если у признака высокая кардинальность, OHE добавит в таблицу огромное количество новых столбцов, что приведет к чрезмерной размерности.

Так как сумма ячеек в новых столбцах всегда будет равна 1, то это вызывает линейную зависимость. Решением является использование параметра drop="first", убирающий один базовый столбец. 

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

X_train_ohe = X_train_rare1.copy()
X_val_ohe = X_val_rare1.copy()

# drop='first' защищает от коллинеарности. handle_unknown='ignore' зануляет все OHE-колонки, если в валидации придет новый класс
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

cols_to_ohe = ['HomePlanet', 'Destination']
train_ohe_arr = ohe.fit_transform(X_train_ohe[cols_to_ohe])
val_ohe_arr = ohe.transform(X_val_ohe[cols_to_ohe])

# Собираем DataFrame обратно
ohe_cols = ohe.get_feature_names_out(cols_to_ohe)
train_ohe_df = pd.DataFrame(train_ohe_arr, columns=ohe_cols, index=X_train_ohe.index)
val_ohe_df = pd.DataFrame(val_ohe_arr, columns=ohe_cols, index=X_val_ohe.index)

# Склеиваем и удаляем старые текстовые столбцы
X_train_ohe = pd.concat([X_train_ohe.drop(columns=cols_to_ohe), train_ohe_df], axis=1)
X_val_ohe = pd.concat([X_val_ohe.drop(columns=cols_to_ohe), val_ohe_df], axis=1)

**4. BinaryEncoder.** Компромиссный метод между первым первым и вторым. Он сначала переводит категории в обычные порядковые числа, а затем преобразует эти числа в двоичный код и раскладывает их по отдельным столбцам.

По сравнению с OHE невероятная экономия пространства для признаков средней и высокой кардинальности.

С другой стороны, напрочь стирает прямую геометрическую интерпретируемость весов модели.

In [ ]:
import category_encoders as ce

X_train_bin = X_train_rare1.copy()
X_val_bin = X_val_rare1.copy()

# handle_unknown='value' настраивает кодировщик выделять под новые неизвестные классы отдельный бинарный паттерн
bin_enc = ce.BinaryEncoder(cols=['Cabin'], handle_unknown='value')

X_train_bin = bin_enc.fit_transform(X_train_bin)
X_val_bin = bin_enc.transform(X_val_bin)

**5. FrequencyEncoder (CountEncoder).** Следует из названия, что этот метод насильно заменяет текстовое имя категории на частоту ее появления в обучающейся выборке.

Такой алгоритм предельно прост в расчетах и не раздувает таблицу. Также отлично помогает деревьям решений отделять популярные группы от редких.

Если две абсолютно разные по смыслу категории встречаются в данных с одинаковой частотой, то кодировщик присвоит им одинаковое число. Модель полностью ослепнет и перестанет виеть между ними разницу.

In [ ]:
import category_encoders as ce

X_train_freq = X_train_rare1.copy()
X_val_freq = X_val_rare1.copy()

# normalize=True преобразует абсолютное количество строк в относительную частоту от 0 до 1
freq_enc = ce.CountEncoder(cols=['HomePlanet', 'Destination'], normalize=True, handle_unknown='value')

X_train_freq = freq_enc.fit_transform(X_train_freq)
X_val_freq = freq_enc.transform(X_val_freq)

**6. TargetEncoder (Mean Encoding).** Метод заменяет название категории средним значением таргета среди всех обьектов этой категории. 

Поэтому чтобы кодировщик не переобучался на слишком редких категориях, применяется метод сглаживания:$$S_i = \lambda(n_i) \cdot \mu_i + (1 - \lambda(n_i)) \cdot \mu_{\text{global}}$$где $\mu_i$ - среднее по категории, $\mu_{\text{global}}$ - глобальное среднее по всему трейну, а вес $\lambda$ зависит от количества строк $n_i$ в этой категории. Чем меньше строк, тем сильнее значение притягивается к глобальной константе.

Это мощнейший алгоритм для высоко кардинальных признаков и деревянных моделей (бустингов).

Но огромная склонность к утечке данных. На валидацию и тест переносятся просто зафиксированные средние.

In [ ]:
import category_encoders as ce

X_train_te = X_train_rare1.copy()
X_val_te = X_val_rare1.copy()

# smoothing регулирует силу m-estimate сглаживания
te_enc = ce.TargetEncoder(cols=['Cabin', 'HomePlanet_Destination'], smoothing=10.0, handle_unknown='value')

# Передаем y_train, так как метод обучается на ответах!
X_train_te = te_enc.fit_transform(X_train_te, y_train)
X_val_te = te_enc.transform(X_val_te)

**7. LeaveOneOutEncoder (LOO).** Модификация TargetEncoder. Необходим чтобы полностью убрать эффект самокодирования (когда обьект сам влияет на среднее значение своей категории), при расчете значения для конкретной $i$-й строки, её собственный таргет $y_i$ вычитается из формулы:$$\hat{x}_i = \frac{\sum_{j \neq i} y_j}{n - 1}$$

Практически полностью убирает эффект переобучения TargetEncoder на обучающейся выборке. Признак получается очень стабильным.

На валидации и тесте работает в точости как обычный TargetEncoder, вель там вычитать чужие ответы нельзя и берется просто накопленное среднее.

In [ ]:
import category_encoders as ce

X_train_loo = X_train_rare1.copy()
X_val_loo = X_val_rare1.copy()

# sigma добавляет небольшой случайный шум к значениям трейна для дополнительной защиты от переобучения
loo_enc = ce.LeaveOneOutEncoder(cols=['Cabin'], sigma=0.05, handle_unknown='value')

X_train_loo = loo_enc.fit_transform(X_train_loo, y_train)
X_val_loo = loo_enc.transform(X_val_loo)

**8. CatBoost Encoding.** Данный метод был придуман Яндексом, он использует онлайн-кодирование по времени. Мысленно перемешивает данные и для каждой текущей строки рассчитывает среднее значение таргета тольпо по тем строкам, которые лежат выше нее.

Абсолютная защита от утечки данных. Модели, обученные на таких счетчиках показывают наивысшую обобщающую способность на незнакомых тестах.

Однако расчеты зависят от случайного стартового перемешивания строк.

In [ ]:
import category_encoders as ce

X_train_cb = X_train_rare1.copy()
X_val_cb = X_val_rare1.copy()

cb_enc = ce.CatBoostEncoder(cols=['Cabin', 'HomePlanet_Destination'], handle_unknown='value')

X_train_cb = cb_enc.fit_transform(X_train_cb, y_train)
X_val_cb = cb_enc.transform(X_val_cb)

**9. Hashing Encoder.** Пропускает текст категории через хэш-функцию и бере остаток от деления на заданное число n_cmponents. Поо сути, это OHE, который принудительно сжимает категории в ограниченное число столбцов.

Если кардинальность чрезмерно большая и улетает в бесконечность этот метод - спасение. Или когда новые категории появляются непрерывно, уме нужно держать в памяти словарии соответсви слов и чисел.

Риск коллизий (разные слова могут получить один хэш и упасть в одну колонку).

In [ ]:
import category_encoders as ce

X_train_hash = X_train_rare1.copy()
X_val_hash = X_val_rare1.copy()

# n_components задает жесткую финальную ширину признака
hash_enc = ce.HashingEncoder(cols=['Cabin'], n_components=8)

X_train_hash = hash_enc.fit_transform(X_train_hash)
X_val_hash = hash_enc.transform(X_val_hash)

**Шаг пятый.** Следующим логичным шагом будет Обработка новых категорий, которых не было в train. А именно тонкости настройки стратегий unknown и параметра handle_unknown. 

**Стратегий 1. Игнорирование.** Если алгоритм видит новое значение в тесте, он делает вид, что его не существует.

Это самый надежный варинат, стандартный способ для OHE. Код никогда не упадет.

Но мы теряем информацию о том, что этот обьект особенный.

Вариант 1. Настройка handle_unknown в OneHotEncoder (sklearn)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# handle_unknown='ignore' - если на тесте придет новая планета, 
# во всех сгенерированных OHE-колонках для этой строки проставятся 0.
ohe_secure = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

# Обучаем на трейне (где редкие категории уже зачищены на Шаге 3)
ohe_secure.fit(X_train_rare1[['HomePlanet']])

**Стратегий 2. Выделение в отдельную корзину (value/unknown).** Кодировщик резервирует специальное числовое значение или отдельный бинарный столбец под названием "неизвестная категория". Любое новое значение на тесте автоматичски приравнивается к этой корзине.

Модель явно видит, что это новая категория и под нее выделен отдельный счетчик/признак.

На этапе fit эта корзина обычно пуста и заполняется искуственно, из-за чего модели бывает трудно подобрать под нее адекватный вес.

Вариант 2. Настройка handle_unknown в OrdinalEncoder (sklearn)

В числовом кодировании нельзя просто проигнорировать значение, ему нужно присвоить какое-то число. Поэтому здесь используется стратегия замены на константу.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# handle_unknown='use_encoded_value' заставляет кодировщик не падать при виде нового значения,
# а unknown_value=-1 принудительно ставит ему число -1.
ord_secure = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

ord_secure.fit(X_train_rare1[['Pclass']])

Вариант 3. Настройка handle_unknown в продвинутых кодировщиках (category_encoders)

В библиотеке category_encoders (для TargetEncoder, CatBoostEncoder, BinaryEncoder) по умолчанию зашит умный параметр handle_unknown='value'.

In [ ]:
import category_encoders as ce

# handle_unknown='value' - если на тесте придет новая каюта, 
# TargetEncoder автоматически подставит вместо неё глобальное среднее значение таргета по всему train!
# (То есть m-estimate сглаживание сработает на максимум)
te_secure = ce.TargetEncoder(cols=['Cabin'], handle_unknown='value', smoothing=10.0)

te_secure.fit(X_train_rare1, y_train)

**Шаг шестой.** Снижение размерности. По факту этот шаг актуальный только если на Шаге 4 использовался OHE. Использутся те же методы что и для числовых признаков 

**1. TruncatedSVD.** Этот метод выполняет усеченное сингулярное разложение (Singular Value Decomposition) матрицы. Находит N наиболее значимы скрытых направлений, которые ударживают максимум дисперсии данных.

Главное отличие от PCA заключается в том, что PCA перед началом сжатия принудительно центрирует данные. Если вычесть среднее из разреженной матрицы, все нулевые ячейки превратятся в мелкие дробные числа, матрица станет плотной, и компьютер моментально исчерпает всю оперативную память. TruncatedSVD работает с данными без центрирования, сохраняя структуру разреженной матрицы в памяти в процессе вычислений.

Работает быстро идеально спроектирован под One-Hot матрицы и текстовые векторы.

Но полностью теряет физический смысл признаков.

In [ ]:
from sklearn.decomposition import TruncatedSVD
import pandas as pd

X_train_svd = X_train_ohe.copy()
X_val_svd = X_val_ohe.copy()

# Находим все OHE-столбцы, которые мы создали (они начинаются с префиксов оригинальных колонок)
ohe_features = [col for col in X_train_svd.columns if col.startswith(('HomePlanet_', 'Destination_'))]

# Задаем количество латентных (скрытых) компонент, в которые хотим сжать признаки
n_comp = 5
svd = TruncatedSVD(n_components=n_comp, random_state=42)

# Обучаем строго на OHE-столбцах трейна, применяем к обоим датасетам
train_svd_components = svd.fit_transform(X_train_svd[ohe_features])
val_svd_components = svd.transform(X_val_svd[ohe_features])

# Оцениваем, какую долю информации удержал алгоритм (explained variance)
print(f"Доля сохраненной дисперсии после сжатия: {svd.explained_variance_ratio_.sum():.4f}")

# Создаем новые числовые столбцы в датасете
for i in range(n_comp):
    X_train_svd[f'OHE_SVD_{i+1}'] = train_svd_components[:, i]
    X_val_svd[f'OHE_SVD_{i+1}'] = val_svd_components[:, i]

# Физически удаляем старые громоздкие OHE-столбцы, заменяя их на компактные компоненты
X_train_svd = X_train_svd.drop(columns=ohe_features)
X_val_svd = X_val_svd.drop(columns=ohe_features)

**2. PCA.** Стандартный метод главных компонент. Подробно рассмотрен на числовых признаках.

Математически эквивалентен SVD на центрированных данных. 

Его можно использовать после OHE только если итоговое количество OHE-столбцов ннебольше и таблица леко помещается в RAM в несжатом плотном виде.

In [ ]:
from sklearn.decomposition import PCA

X_train_pca_cat = X_train_ohe.copy()
X_val_pca_cat = X_val_ohe.copy()

# Допустимо только если len(ohe_features) небольшая и RAM позволяет уплотнить матрицу
pca_cat = PCA(n_components=3, random_state=42)

train_pca_cat = pca_cat.fit_transform(X_train_pca_cat[ohe_features])
val_pca_cat = pca_cat.transform(X_val_pca_cat[ohe_features])

# Записываем компоненты и дропаем разреженные хвосты
for i in range(3):
    X_train_pca_cat[f'OHE_PCA_{i+1}'] = train_pca_cat[:, i]
    X_val_pca_cat[f'OHE_PCA_{i+1}'] = val_pca_cat[:, i]

X_train_pca_cat = X_train_pca_cat.drop(columns=ohe_features)
X_val_pca_cat = X_val_pca_cat.drop(columns=ohe_features)

Ура! Две трети пути пройдено. Осталось только проработать текстовые признаки. Будем идти по таким шагам:
1. Ручное извлечение признаков
2. Статистический анализ (EDA)
3. Предобработка текста
4. Классические методы векторизации
5. Снижение размерности после векторизации
6. Эмбеддинг на уровне слов (Статичные векторы)
7. Эмбеддинг предложений и документов (Контекстные векторы)
8. Тематическое моделирование
9. Обработка специфичных крайних случаев

**Шаг первый.** Ручной Feature Engineering для тестовых данных.

Сырой текст содержит в себе огромный пласт скрытой информации, которая полностью уничтожается при стандартной предобработке. Если человек написал слова КАПСОМ (заглавными буквами), поставил кучу восклицательных знаков (!!!) или цифр, то после приведения к нижнему регистру и удаления пунктуации эта информация сотрется навсегда. Но именно эти параметры часто сильнее всего коррелируют с таргетом.

Будем извлекать 5 метрик:
1. *Длина текста (количество слов и количество символов).* Позволяет модели понять обьем информации. Короткие сообщения часто не несут в себе никакую глубокую информацию. А длинные текста могут указывать на подробное описание проблемы.
2. *Количество заглавных букв.* Отлично улавливает эмоции. Если в тексте много заглавных букв, это маркер сильного недовольства.
3. *Количество знаков препинания.* Множество восклицатльных знаков указывает на эмоции, а вопросительные - на наличие неразрешенных вопросов
4. *Количество цифр.* Маркер, показывает присутствует ли в тексте точные числовые данные
5. *Анализ тональности (Sentiment Score).* Позволяет перевести текс в непрерывную числовую шкалу от -1 (абсолютно негативно) до 1 (абсолютно позитивно).

In [ ]:
# !pip install textblob

import pandas as pd
import numpy as np
import string
from textblob import TextBlob

X_train_text_features = X_train_replace.copy()
X_val_text_features = X_val_replace.copy()

text_col = 'Text_Column'

# Защита: заполняем пропуски в тексте пустой строкой, чтобы строковые методы не падали
X_train_text_features[text_col] = X_train_text_features[text_col].fillna("").astype(str)
X_val_text_features[text_col] = X_val_text_features[text_col].fillna("").astype(str)

# Функция для ручного извлечения признаков из одной строки
def extract_manual_meta(df, col_name):
    # 1. Длина текста в символах
    df[f'{col_name}_char_len'] = df[col_name].apply(len)
    
    # 2. Длина текста в словах
    df[f'{col_name}_word_count'] = df[col_name].apply(lambda x: len(x.split()))
    
    # 3. Количество заглавных букв
    df[f'{col_name}_uppercase_count'] = df[col_name].apply(lambda x: sum(1 for char in x if char.isupper()))
    
    # 4. Количество цифр
    df[f'{col_name}_digits_count'] = df[col_name].apply(lambda x: sum(1 for char in x if char.isdigit()))
    
    # 5. Количество знаков препинания
    # string.punctuation содержит строку со всеми стандартными спецсимволами: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
    df[f'{col_name}_punct_count'] = df[col_name].apply(lambda x: sum(1 for char in x if char in string.punctuation))
    
    # 6. Расчет Sentiment Score (Тональность: от -1 до 1)
    # x.sentiment.polarity возвращает число, где < 0 - негатив, 0 - нейтрально, > 0 - позитив
    df[f'{col_name}_sentiment'] = df[col_name].apply(lambda x: TextBlob(x).sentiment.polarity if x.strip() else 0.0)
    
    return df

# Применяем генерацию признаков зеркально и к трейну, и к валидации
X_train_text_features = extract_manual_meta(X_train_text_features, text_col)
X_val_text_features = extract_manual_meta(X_val_text_features, text_col)

# Посмотрим на результат генерации
new_meta_cols = [c for c in X_train_text_features.columns if c.startswith(text_col + '_')]
print(f"Успешно сгенерированы ручные текстовые признаки: {new_meta_cols}")
print(X_train_text_features[[text_col] + new_meta_cols].head(3))

**Шаг второй.** Статистический анализ текста (EDA). После того как на первом шаге мы замерили базовые структурные свойства сырого текста (длину, капсы, пунктуацию), нам необходимо провести лингвистический анализ самого словаря. Это нужно, чтобы понять объем текстового разнообразия и настроить будущие векторы.

Будем рассматривать *количество уникальных слов*, *частота слов и закон Ципфа* и *автоматическое определение языка текста*.

1. **Количество уникальных слов (Размер сырого словаря).**
Показывает богатство языка в датасете. На этом этапе мы вычисляем количество уникальных токенов. Если размер словаря составляет десятки тысяч слов, это сигнал о том, что классическая OHE-векторизация (Bag of Words) превратит таблицу в гигантский, тяжелый и неэффективный массив, и нам точно потребуются продвинутые методы (TF-IDF с ограничением фичей или эмбеддинги).

2. **Частота слов и Закон Ципфа.**
Закон Ципфа (Zipf Law) - это фундаментальное правило эмпирической лингвистики, которое гласит: если расположить все слова языка по убыванию частоты их использования, то частота любого слова будет обратно пропорциональна его порядковому номеру (рангу).

Что это значит на практике: В любом текстовом датасете есть крошечная кучка частых слов (предлоги, союзы, местоимения), которые занимают до 50% всего текста, но несут нулевую пользу для обучения моделей. На другом полюсе находится бесконечный "длинный хвост" слов, которые встречаются всего по 1 разу на миллион строк. Этот анализ помогает обосновать необходимость удаления стоп-слов и редких токенов на следующем шаге.

3. **Автоматическое определение языка текста.**
Если текст мультиязычный, это критическая точка ветвления. Лемматизаторы и списки стоп-слов жестко привязаны к конкретному языку. Алгоритм, настроенный для английского, превратит русский текст в цифровой шум. С помощью легких библиотек мы должны проверить однородность языка в колонке.

In [ ]:
# !pip install langdetect

import pandas as pd
import numpy as np
from collections import Counter
import re
from langdetect import detect, DetectorFactory
import matplotlib.pyplot as plt
import seaborn as sns

# Фиксируем сид для воспроизводимости детектора языка
DetectorFactory.seed = 42

X_train_text_eda = X_train_replace.copy()
text_col = 'Text_Column'

# Базовая очистка от NaN для корректной работы строковых функций
raw_texts = X_train_text_eda[text_col].fillna("").astype(str).tolist()

#1. РАСЧЕТ РАЗМЕРА СЛОВАРЯ И ЧАСТОТНОСТИ СЛОВ
all_words = []
for text in raw_texts:
    # Выделяем только слова (буквенные символы), приводя к нижнему регистру
    words = re.findall(r'[a-zA-Zа-яА-ЯёЁ]+', text.lower())
    all_words.extend(words)

word_counts = Counter(all_words)
total_tokens = len(all_words)
unique_words = len(word_counts)
)
print(f"Статистический анализ для колонки: '{text_col}'")
print(f"• Всего слов в датасете (токенов): {total_tokens}")
print(f"• Размер сырого словаря (уникальных слов): {unique_words}")

# Выводим топ-15 самых частых слов (демонстрация закона Ципфа)
print("\n• Топ-15 самых частых слов (кандидаты в стоп-слова):")
top_words = word_counts.most_common(15)
for word, count in top_words:
    pct = (count / total_tokens) * 100
    print(f"  - '{word}': {count} раз ({pct:.2f}%)")

#2. ВИЗУАЛИЗАЦИЯ ЗАКОНА ЦИПФА
plt.figure(figsize=(10, 4))
counts_values = [count for word, count in word_counts.most_common(50)]
sns.lineplot(x=range(1, len(counts_values) + 1), y=counts_values, marker="o", color="crimson")
plt.title("Распределение частоты слов (Закон Ципфа)")
plt.xlabel("Ранг слова (от самого частого к менее частому)")
plt.ylabel("Абсолютная частота появления")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

#3. АВТОМАТИЧЕСКОЕ ОПРЕДЕЛЕНИЕ ЯЗЫКА (Семплирование для скорости)
# Проверять миллион строк через langdetect долго, берем случайную выборку из 300 непустых строк
sampled_texts = [t for t in raw_texts if t.strip()]
if len(sampled_texts) > 300:
    sampled_texts = np.random.choice(sampled_texts, size=300, replace=False)

languages = []
for text in sampled_texts:
    try:
        languages.append(detect(text))
    except:
        languages.append("unknown")

lang_counts = Counter(languages)
print("• Распределение языков в выборке текстов:")
for lang, count in lang_counts.items():
    lang_pct = (count / len(languages)) * 100
    print(f"Код языка [{lang}]: {count} строк ({lang_pct:.2f}%)")

**Шаг третий.** Предобработка текста (Очистка и нормализация). Цель - убрать из текстовой строки весь структурный, кодовый и лингвистический шум, оставив только чистые смысловые единицы в базовой форме. Если этот этап пропустить или сделать небрежно, словарь раздуется в тысячи раз из-за знаков препинания и разных окончаний одного и того же слова, что полностью сломает классические векторы.

Будем преобразовывать текст в несколько этапов:
1. **Удаление HTML-тегов.** Если тексты парсились из веба, там будут конструкции вида $<br>, <div>, &amp;$. Модели могут принять их за важные слова, поэтому их нужно вычищать регулярными выражениями.
2. **Нижний регистр.** Компьютер считает слова "Бег", "бег" и "БЕГ" тремя абсолютно разными токенами. Приведение к .lower() схлопывает этот дублирующийся шум.
3. **Удаление пунктуации, спецсимволов и цифр.** Знаки препинания (!, ?, ,) и цифры удаляются, чтобы изолировать чистые слова.
4. **Нормализация Unicode.** Символы вроде некоторых букв с диакритическими знаками или специфические кодировки пробелов могут выглядеть одинаково, но иметь разные байт-коды. Метод unicodedata.normalize('NFKC', text) приводит их к единому стандарту.
5. **Токенизаци.** Процесс расщепления сплошной строки текста на массив отдельных слов. Стандартный .split() плох тем, что не умеет грамотно обрабатывать сложные переносы и дефисы. Поэтому используют токенизаторы из специализированных библиотек (nltk или spacy).
6. **Удаление стоп-слов.** Союзы, предлоги и местоимения (и, в, на, the, and) встречаются в текстах чаще всего (вспоминаем закон Ципфа из Шага 2). Они не несут уникального смысла для классификации (например, отзыв позитивный или негативный), поэтому их вырезают по готовым спискам.
7. **Стемминг или Лемматизация.**

Стемминг (Stemming) работает по жестким правилам и просто отрезает от слова окончания и суффиксы. "бегущий", "бегала", "бег" $\to$ превращаются в обрубок "бег". Не знает правил языка. Слово "хороший" и "лучший" он никогда не свяжет. Подходит для грубого быстрого поиска. Лемматизация (Lemmatization) же приводит слово к его исходной словарной форме (инфинитив для глаголов, именительный падеж/единственное число для существительных). "бегущий" $\to$ "бежать", "людьми" $\to$ "человек". Математически и лингвистически точный метод. Идеален для построения качественных признаков. Для английского стандартами являются WordNetLemmatizer (nltk) или spaCy. Для русского языка - pymorphy3 или MyStem.

In [ ]:
#!pip install nltk pymorphy3

import pandas as pd
import numpy as np
import re
import string
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pymorphy3

# Скачиваем необходимые лингвистические пакеты NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

X_train_text_clean = X_train_replace.copy()
X_val_text_clean = X_val_replace.copy()

text_col = 'Text_Column'

# Инициализируем русский лемматизатор и список стоп-слов
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))

# Монолитная функция для Шага 3 (Предобработка текста)
def preprocess_text_pipeline(text):
    if not isinstance(text, str) or pd.isna(text):
        return ""
    
    # 1. Удаление HTML тегов
    text = re.sub(r'<[^>]+>', '', text)
    
    # 2. Нормализация Unicode (NFKC кодирование)
    text = unicodedata.normalize('NFKC', text)
    
    # 3. Перевод в нижний регистр
    text = text.lower()
    
    # 4. Удаление пунктуации, спецсимволов и цифр (оставляем только буквы и пробелы)
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ\s]', ' ', text)
    
    # 5. Токенизация (разбиение на слова через NLTK)
    tokens = word_tokenize(text)
    
    # 6. Удаление стоп-слов и Лемматизация за один проход
    cleaned_lemmas = []
    for token in tokens:
        # Убираем лишние пробелы по краям
        token = token.strip()
        if token and token not in russian_stopwords:
            # Приводим к нормальной форме (лемме) через pymorphy3
            lemma = morph.parse(token)[0].normal_form
            cleaned_lemmas.append(lemma)
            
    # Собираем очищенные леммы обратно в единую строку через пробел
    return " ".join(cleaned_lemmas)

# Применяем пайплайн очистки к трейну и валидации
X_train_text_clean[f'{text_col}_clean'] = X_train_text_clean[text_col].apply(preprocess_text_pipeline)
X_val_text_clean[f'{text_col}_clean'] = X_val_text_clean[text_col].apply(preprocess_text_pipeline)

print("Сравнение До и После предобработки:")
print("Оригинал:", X_train_text_clean[text_col].iloc[0])
print("Результат Шага 3:", X_train_text_clean[f'{text_col}_clean'].iloc[0])

**Шаг четвертый.** Классические методы векторизации или Статистический подход. На этом этапе необходимо взять уже обработанную строку текста, которую получили после шага три. И эта строка на шаге 4 превращается в матрицу чисел, понятную математическим моделям. В отличие от сложны нейросетевых эмбеддингов, классические методы опираются на статистику встечаемости токенов.

При использовании классических методов векторизации (CountVectorizer, TfidfVectorizer) без ограничений, словарь может раздуться до сотен тысяч колонок из-за опечаток, редких слов и имен. Чтобы этого избежать, перед изучением методов зафиксируем 3 критических параметра фильтрации словаря, которые есть в sklearn:
* **min_df (Minimum Document Frequency).** Игнорирует слова, которые встречаются реже, чем в N документах датасета (или задается долей, например 0.005 - меньше 0.5% документов). Помогает срезать шум.
* **max_df (Maximum Document Frequency).** Игнорирует слова, которые встречаются чаще, чем, например, в 90% документов (max_df=0.90). Помогает убрать слишком общие слова, если их пропустил список стоп-слов.
* **max_features.** Жестко ограничивает размерность, оставляя в матрице только топ-N самых частых слов во всем датасете.

**1. Bag of Words (Мешок слов/CountVectorizer).** Создает огромную матрицу, где каждый столбец - это уникальное слово из словаря, а каждая строка - один документ. На пересечении ставится абсолютное количество раз, которое это слово встретилось в данном тексте.

У этого метода предельно простая логика. Идеально работает в связке с Наивным Байесовским классификатором (Naive Bayes) для быстрой фильтрации спама или тематик.

Игнорирует контекст и порядок слов (предложения "я не люблю кошек, а люблю собак" и "я не люблю собак, а люблю кошек" будут иметь абсолютно одинаковые векторы). Длинные тексты искусственно получают огромные значения просто за счет своего объема.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# В качестве входящих данных берем результаты Шага 3 (столбец 'Text_Column_clean')
X_train_bow = X_train_text_clean.copy()
X_val_bow = X_val_text_clean.copy()

# Настраиваем фильтрацию словаря: берем слова, встретившиеся минимум в 2 документах
cv = CountVectorizer(min_df=2, max_features=1000)

# fit_transform делаем строго на Train, на Validation - только transform!
train_bow_sparse = cv.fit_transform(X_train_bow['Text_Column_clean'])
val_bow_sparse = cv.transform(X_val_bow['Text_Column_clean'])

# Для шпоры: превращаем разреженную матрицу в читаемый DataFrame
train_bow_df = pd.DataFrame(train_bow_sparse.toarray(), columns=cv.get_feature_names_out(), index=X_train_bow.index)
print("Пример матрицы Bag of Words:")
print(train_bow_df.head(3))

**2. TF-IDF (TfidfVectorizer).** Рассчитывает статистический вес каждого слова, отражающий его важность в конкретном документе относительно всей выборки. Состоит из двух перемножаемых частей:
* TF (Term Frequency): Частота слова внутри конкретного документа (сколько раз встретилось слово, поделенное на общее число слов в этой строке).
* IDF (Inverse Document Frequency): Инверсия частоты слова во всем датасете. Логарифм от общего числа документов, поделенного на количество документов, содержащих это слово.

Чем чаще слово встречается во всем датасете, тем ближе его IDF к нулю.$$\text{TF-IDF} = \text{TF} \times \text{IDF}$$

Нивелирует проблему длины текста. Ультра-частые слова (которые не отсеклись на этапе чистки) автоматически получают низкий вес, а уникальные маркеры темы (например, узкоспециализированные термины) - высокий вес.

Но по-прежнему полностью игнорирует порядок слов.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_tfidf = X_train_text_clean.copy()
X_val_tfidf = X_val_text_clean.copy()

# sublinear_tf=True применяет логарифмическое сглаживание к TF (1 + log(TF)), что защищает от доминирования длинных текстов
tfidf = TfidfVectorizer(min_df=3, max_df=0.9, sublinear_tf=True, max_features=1500)

train_tfidf_sparse = tfidf.fit_transform(X_train_tfidf['Text_Column_clean'])
val_tfidf_sparse = tfidf.transform(X_val_tfidf['Text_Column_clean'])

train_tfidf_df = pd.DataFrame(train_tfidf_sparse.toarray(), columns=tfidf.get_feature_names_out(), index=X_train_tfidf.index)

**3. N-grams (N-граммы).** Позволяет частично решить проблему потери контекста. Вместо того чтобы резать текст строго по одному слову (униграммы), алгоритм склеивает цепочки из соседних слов. Например, при ngram_range=(1,2) для фразы "не люблю" создадутся токены: не, люблю (униграммы) и единый токен не_люблю (биграмма).

Ловит устойчивые словосочетания, отрицания и меняющие смысл связки слов.

Огромная лавинообразная скорость роста словаря. Включение триграмм способно увеличить размерность матрицы в десятки раз.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_ng = X_train_text_clean.copy()
X_val_ng = X_val_text_clean.copy()

# ngram_range=(1, 2) означает собирать и отдельные леммы, и пары соседних слов (биграммы)
tfidf_ng = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=2000)

train_ng_sparse = tfidf_ng.fit_transform(X_train_ng['Text_Column_clean'])
val_ng_sparse = tfidf_ng.transform(X_val_ng['Text_Column_clean'])

**4. Hashing Vectorizer (HashingVectorizer).** NLP-аналог категориального HashingEncoder. Вместо построения и удержания в оперативной памяти словаря (словарь $\to$ индекс), он применяет хэш-функцию к тексту токена и сразу вычисляет его индекс в матрице заданной фиксированной ширины.

Практически не потребляет оперативную память при обработке бесконечных потоков данных (Out-of-core learning). Идеален для терабайтных логов или стриминга данных.

Из-за коллизий хэширования невозможно сделать .get_feature_names_out(), то есть вы никогда не узнаете, какой столбец матрицы отвечает за какое конкретно человеческое слово.

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

X_train_hash = X_train_text_clean.copy()
X_val_hash = X_val_text_clean.copy()

# n_features=512 жестко фиксирует, что любой текст сожмется ровно в 512 столбцов разреженной матрицы
hash_vec = HashingVectorizer(n_features=512, alternate_sign=False)

train_hash_sparse = hash_vec.fit_transform(X_train_hash['Text_Column_clean'])
val_hash_sparse = hash_vec.transform(X_val_hash['Text_Column_clean'])

**Шаг шестой.** Снижение размерности после классической векторизации. 

Для этого используется метод LSA (Latent Semantic Analysis / Латентно-семантический анализ). Математически LSA представляет собой не что иное, как TruncatedSVD (Усеченное сингулярное разложение), примененное к матрице TF-IDF.

**Вариант 1.** Сжатие и выделение скрытого смысла через TruncatedSVD (Математический LSA).

In [ ]:
from sklearn.decomposition import TruncatedSVD
import pandas as pd

# Задаем желаемое количество латентных компонент (тем). Обычно берут от 10 до 100 в зависимости от объема текстов.
n_topics = 10
svd_nlp = TruncatedSVD(n_components=n_topics, random_state=42)

# Обучаем строго на разреженной TF-IDF матрице ТРЕЙНА, применяем к обоим датасетам
train_lsa_features = svd_nlp.fit_transform(train_tfidf_sparse)
val_lsa_features = svd_nlp.transform(val_tfidf_sparse)

# Посмотрим, сколько информации (дисперсии) смогли удержать эти компоненты
print(f"• Доля сохраненной дисперсии: {svd_nlp.explained_variance_ratio_.sum():.4f}")

# Оформляем в виде понятных числовых колонок для финального DataFrame
lsa_cols = [f'SVD_Topic_{i+1}' for i in range(n_topics)]
train_lsa_df = pd.DataFrame(train_lsa_features, columns=lsa_cols, index=X_train_text_clean.index)
val_lsa_df = pd.DataFrame(val_lsa_features, columns=lsa_cols, index=X_val_text_clean.index)

print(f"• Матрица успешно сжата с {train_tfidf_sparse.shape[1]} признаков до {n_topics} компонент!")

**Вариант 2.** Визуализация текстовых кластеров на плоскости через UMAP.

Если стоит задача увидеть глазами, как тексты группируются по тема или классам.

In [ ]:
# !pip install umap-learn
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# Сжимаем данные до 2D координат для построения графика
umap_nlp = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)

# Проецируем LSA-признаки трейна на плоскость
train_umap_2d = umap_nlp.fit_transform(train_lsa_features)

# Строим красивый scatter-plot ( hue=y_train раскрасит точки по целевым классам )
plt.figure(figsize=(10, 6))
sns.scatterplot(x=train_umap_2d[:, 0], y=train_umap_2d[:, 1], hue=y_train, palette='coolwarm', alpha=0.6)
plt.title("2D проекция текстового датасета через UMAP")
plt.xlabel("UMAP координата 1")
plt.ylabel("UMAP координата 2")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

Стоит отметить, что как и с числовыми данными шаги 6 и 7 идут не последовательно, а параллельно. Это два альтернативных метода

**Шаг шестой.** Эмбиддинги на уровне слов.

В отличие от классических методов, которые смотрят только на голые частоты букв и не понимают скрытого смысла слов, эмбеддинги опираются на гипотезу, что слова, которые часто встречаются в одинаковых контекстах, имеют близкие смысловые значения. 

Вместо разреженной матрицы шириной в размер всего словаря, этот подход сопоставляет каждому слову плотный вектор фиксированной размерности, где геометрия пространства отражает логические связи. В таком пространстве работает лингвистическая арифметика: король - мужчина + женщина = королева.

Подробно разберем три главных классических алгоритма векторных представлений слов:

In [ ]:
# !pip install gensim

**1. Word2Vec (Разработан Google под руководством Томаша Миколова).** Этот метод имеет две классические архитектуры:
* **CBOW (Continuous Bag of Words):** Модель предсказывает текущее центральное слово по его окружающему контексту (соседним словам).
* **Skip-gram:** Модель берет одно центральное слово и, наоборот, пытается предсказать вероятность появления контекстных слов вокруг него. (Skip-gram работает лучше на редких словах, но обучается дольше).

Метод отлично ловит синонимы и базовые ассоциации. Сжимает смыслы без раздувания таблицы.

Однако векторы статичны и не учитывают контекст омонимов. Для Word2Vec слово "лук" (оружие) и "лук" (овощ) получат один и тот же усредненный вектор. Кроме того, если слова нет в обучающем словаре, для него невозможно получить эмбеддинг (OOV - Out of Vocabulary).

In [ ]:
# !pip install gensim

import numpy as np
import pandas as pd
from gensim.models import Word2Vec

X_train_w2v = X_train_text_clean.copy()
X_val_w2v = X_val_text_clean.copy()

# Word2Vec на вход принимает список списков токенов. Бьем строки по пробелам обратно на массивы слов
train_tokenized = [text.split() for text in X_train_w2v['Text_Column_clean']]
val_tokenized = [text.split() for text in X_val_w2v['Text_Column_clean']]

# 1. Обучаем модель Word2Vec строго на тренировочных текстах
# vector_size=100 - длина плотного вектора для каждого слова
# window=5 - размер контекстного окна (5 слов слева и справа)
# min_count=2 - игнорировать слова, которые встретились во всем датасете только 1 раз
w2v_model = Word2Vec(sentences=train_tokenized, vector_size=100, window=5, min_count=2, workers=4, seed=42)

# 2. Функция для агрегации векторов слов в единый вектор документа ( Mean Pooling )
def text_to_vector(token_list, model, vector_size):
    # Фильтруем слова, которые есть в обученном словаре Word2Vec
    valid_vectors = [model.wv[word] for word in token_list if word in model.wv]
    
    # Если в тексте не оказалось ни одного знакомого слова, возвращаем нулевой вектор
    if len(valid_vectors) == 0:
        return np.zeros(vector_size)
    
    # Находим среднее арифметическое по всем осям для векторов слов
    return np.mean(valid_vectors, axis=0)

# 3. Переводим все тексты трейна и валидации в плотные матрицы векторов
train_w2v_arr = np.array([text_to_vector(tokens, w2v_model, 100) for tokens in train_tokenized])
val_w2v_arr = np.array([text_to_vector(tokens, w2v_model, 100) for tokens in val_tokenized])

# 4. Превращаем массивы в DataFrame с красивыми именами колонок и склеиваем
w2v_cols = [f'W2V_dim_{i+1}' for i in range(100)]
train_w2v_df = pd.DataFrame(train_w2v_arr, columns=w2v_cols, index=X_train_w2v.index)
val_w2v_df = pd.DataFrame(val_w2v_arr, columns=w2v_cols, index=X_val_w2v.index)

print(f"• Размерность полученной матрицы эмбеддингов: {train_w2v_df.shape}")
print(train_w2v_df.head(3))

**2. GloVe (Global Vectors для представления слов от Stanford).** В отличие от Word2Vec, который бежит локальным окном по тексту, GloVe обучается на глобальной матрице совместной встречаемости слов (co-occurrence matrix) во всем корпусе текстов.

Математически более стабилен на огромных текстовых базах (например, Википедии), так как учитывает логарифмы отношений вероятностей появления слов.

In [ ]:
import gensim.downloader as api
import numpy as np
import pandas as pd

X_train_glove = X_train_text_clean.copy()
X_val_glove = X_val_text_clean.copy()

# Скачиваем официальные векторы GloVe (размерность 100, обучены на текстах Википедии/Твиттера)
# Это займет некоторое время при первом скачивании
glove_model = api.load("glove-wiki-gigaword-100")

def get_glove_matrix(df_column, model, vector_size=100):
    matrix = np.zeros((len(df_column), vector_size))
    for idx, text in enumerate(df_column):
        tokens = str(text).split()
        # Ищем векторы слов в GloVe-словаре
        valid_vectors = [model[word] for word in tokens if word in model]
        if len(valid_vectors) > 0:
            matrix[idx] = np.mean(valid_vectors, axis=0)
    return matrix

train_glove_arr = get_glove_matrix(X_train_glove['Text_Column_clean'], glove_model)
val_glove_arr = get_glove_matrix(X_val_glove['Text_Column_clean'], glove_model)

glove_cols = [f'GloVe_dim_{i+1}' for i in range(100)]
train_glove_df = pd.DataFrame(train_glove_arr, columns=glove_cols, index=X_train_glove.index)
val_glove_df = pd.DataFrame(val_glove_arr, columns=glove_cols, index=X_val_glove.index)

**3. FastText (Разработан Facebook).** Эволюция Word2Vec. Алгоритм бьет каждое слово на кусочки - символьные n-граммы (subwords). Например, слово бегущий будет разобрано на подстроки: бег, егу, гущ и т.д.

Решает проблему OOV (Out-of-Vocabulary) и опечаток. Если модель встретит неизвестное ей слово бежавший, она соберет его вектор из векторов знакомых n-грамм (беж, авш).

Требует существенно больше оперативной памяти и дискового пространства для хранения подсловарных индексов.

In [ ]:
from gensim.models import FastText
import numpy as np
import pandas as pd

X_train_ft = X_train_text_clean.copy()
X_val_ft = X_val_text_clean.copy()

train_ft_tokens = [text.split() for text in X_train_ft['Text_Column_clean']]
val_ft_tokens = [text.split() for text in X_val_ft['Text_Column_clean']]

# Обучаем модель FastText
# min_n=3, max_n=6 - задают размеры символьных n-грамм для разбиения слов внутри
ft_model = FastText(sentences=train_ft_tokens, vector_size=100, window=5, min_count=2, min_n=3, max_n=6, seed=42)

def get_fasttext_matrix(token_list, model, vector_size=100):
    matrix = np.zeros((len(token_list), vector_size))
    for idx, tokens in enumerate(token_list):
        # ВАЖНОЕ ОТЛИЧИЕ: FastText никогда не выдаст ошибку OOV! 
        # Если слова нет в словаре, он сам соберет его вектор по символьным n-граммам.
        valid_vectors = [model.wv[word] for word in tokens]
        if len(valid_vectors) > 0:
            matrix[idx] = np.mean(valid_vectors, axis=0)
    return matrix

train_ft_arr = get_fasttext_matrix(train_ft_tokens, ft_model)
val_ft_arr = get_fasttext_matrix(val_ft_tokens, ft_model)

ft_cols = [f'FT_dim_{i+1}' for i in range(100)]
train_ft_df = pd.DataFrame(train_ft_arr, columns=ft_cols, index=X_train_ft.index)
val_ft_df = pd.DataFrame(val_ft_arr, columns=ft_cols, index=X_val_ft.index)

print("Матрица FastText успешно создана для Train и Validation выборки!")

**Шаг седьмой.** Эмбеддинги на уровне предложений и документов.

Если на предыдущем шаге мы работали со статическими векторами отдельных слов и склеивали их вручную, то теперь мы разберем продвинутые методы, которые сразу переводят целую текстовую строку в фиксированный плотный вектор. Они способны улавливать сложную контекстную синергию и порядок слов в предложении.

In [ ]:
#!pip install sentence-transformers transformers torch

**1. Doc2Vec (Расширение Word2Vec от Google).** В отличие от Word2Vec, здесь к контекстному окну слов добавляется еще один уникальный токен - ID самого документа (Paragraph ID). Модель обучается предсказывать слова, учитывая не только соседние токены, но и вектор всей строки целиком.

Работает быстро на обычных процессорах (CPU), не требует видеокарт, умеет хорошо обобщать смысл коротких текстов.

Векторы по-прежнему статические (не учитывают тонкую контекстную зависимость омонимов).

In [ ]:
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

X_train_d2v = X_train_text_clean.copy()
X_val_d2v = X_val_text_clean.copy()

# 1. Doc2Vec требует специальный формат TaggedDocument (токены + уникальный тег-индекс строки)
train_tagged = [TaggedDocument(doc.split(), [f"train_{idx}"]) for idx, doc in enumerate(X_train_text_clean['Text_Column_clean'])]

# 2. Обучаем модель Doc2Vec
d2v_model = Doc2Vec(documents=train_tagged, vector_size=100, window=5, min_count=2, workers=4, epochs=20, seed=42)

# 3. Накапливаем векторы (инференс) для трейна и валидации
train_d2v_arr = np.array([d2v_model.infer_vector(text.split()) for text in X_train_d2v['Text_Column_clean']])
val_d2v_arr = np.array([d2v_model.infer_vector(text.split()) for text in X_val_d2v['Text_Column_clean']])

# 4. Собираем DataFrame
d2v_cols = [f'Doc2Vec_dim_{i+1}' for i in range(100)]
train_d2v_df = pd.DataFrame(train_d2v_arr, columns=d2v_cols, index=X_train_d2v.index)
val_d2v_df = pd.DataFrame(val_d2v_arr, columns=d2v_cols, index=X_val_d2v.index)

**2. Sentence-BERT (SBERT).** Модификация классической нейросети BERT, обученная на сиамских (двойных) архитектурах. Она спроектирована специально для того, чтобы на выходе выдавать семантически точные, сопоставимые между собой векторы для целых предложений.

Это промышленный стандарт для поиска похожих текстов, дедупликации и семантического поиска. Фантастическое качество понимания контекста.

Требует много вычислительных ресурсов (для больших датасетов без GPU будет работать медленно).

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

X_train_sbert = X_train_text_clean.copy()
X_val_sbert = X_val_text_clean.copy()

# Загружаем легковесную, но сильную мультиязычную модель SBERT
print("Загрузка предобученной модели Sentence-BERT...")
sbert_model = SentenceTransformer('sentence-transformers/LaBSE')

# Кодируем тексты в плотные эмбеддинги
# По умолчанию LaBSE возвращает векторы размерности 768
print("Кодирование текстов...")
train_sbert_arr = sbert_model.encode(X_train_sbert['Text_Column_clean'].tolist(), show_progress_bar=True)
val_sbert_arr = sbert_model.encode(X_val_sbert['Text_Column_clean'].tolist(), show_progress_bar=True)

sbert_cols = [f'SBERT_dim_{i+1}' for i in range(train_sbert_arr.shape[1])]
train_sbert_df = pd.DataFrame(train_sbert_arr, columns=sbert_cols, index=X_train_sbert.index)
val_sbert_df = pd.DataFrame(val_sbert_arr, columns=sbert_cols, index=X_val_sbert.index)

**3. Universal Sentence Encoder (USE от Google).** Тяжелая нейросетевая модель от Google, использующая архитектуру Transformer или глубокие сети (DAN). Направлена на кодирование предложений сразу под задачи трансферного обучения.

Очень стабильные результаты на мультиязычных текстах "из коробки".

**4. BERT / RoBERTa / Transformers (CLS-токен).** Использование сырой предобученной языковой модели (LLM). Текст подается на вход, и в качестве эмбеддинга всего предложения берется скрытое состояние самого первого служебного токена - [CLS] (Classification token), который аккумулирует в себе контекст всей строки при прохождении через слои внимания (Attention).

Самый мощный, кастомный подход для последующего построения нейросетей.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd

X_train_bert = X_train_text_clean.copy()
X_val_bert = X_val_text_clean.copy()

# Загружаем токенизатор и саму базовую модель (например, легкий DistilBERT)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
bert_model = AutoModel.from_pretrained("distilbert-base-multilingual-cased")

# Переводим вычисления на видеокарту, если она есть
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = bert_model.to(device)

def get_bert_cls_embeddings(texts, tokenizer, model, batch_size=32):
    model.eval()
    all_embeddings = []
    
    # Идем по датасету батчами для экономии оперативной памяти
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Токенизируем с забивкой (padding) до максимальной длины в батче
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # outputs.last_hidden_state содержит матрицу размера [batch_size, seq_len, hidden_dim]
        # Берем [:, 0, :] - это состояние первого [CLS] токена для всех строк батча
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
        
    return np.vstack(all_embeddings)

# Кодируем (передаем сырые списки строк)
train_bert_arr = get_bert_cls_embeddings(X_train_bert['Text_Column_clean'].tolist(), tokenizer, bert_model)
val_bert_arr = get_bert_cls_embeddings(X_val_bert['Text_Column_clean'].tolist(), tokenizer, bert_model)

bert_cols = [f'BERT_CLS_{i+1}' for i in range(train_bert_arr.shape[1])]
train_bert_df = pd.DataFrame(train_bert_arr, columns=bert_cols, index=X_train_bert.index)
val_bert_df = pd.DataFrame(val_bert_arr, columns=bert_cols, index=X_val_bert.index)

**Шаг восьмой.** Тематическое моделирование (Алгоритмы без учителя). 

В отличие от предыдущих шагов, где мы пытались превратить текст в геометрический вектор, тематическое моделирование пытается решить другую задачу: найти скрытые темы внутри корпуса текстов и определить, в какой пропорции каждая тема представлена в конкретном документе.

Это алгоритмы обучения без учителя. Модель не знает заранее, о чем тексты. Она находит математические паттерны совместной встречаемости слов и говорит: "Этот текст на 70% состоит из Темы №1 (где главные слова: космос, корабль, полет) и на 30% из Темы №2 (билет, цена, каюта)".

**1. LDA (Latent Dirichlet Allocation / Латентное размещение Дирихле).** Это вероятностная генеративная модель. Она предполагает, что каждый документ - это смесь распределений над скрытыми темами, а каждая тема - это распределение над словами словаря (используется распределение Дирихле). Алгоритм пытается восстановить эти скрытые распределения, чаще всего с помощью сэмплирования Гиббса.

Строгая статистическая основа, прекрасная интерпретируемость (можно прямо посмотреть ключевые слова для каждого топика).

Требует ручного подбора количества тем (K), работает относительно медленно на огромных текстах.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd
import numpy as np

X_train_lda = X_train_text_clean.copy()
X_val_lda = X_val_text_clean.copy()

# Задаем желаемое количество скрытых тем
n_components_topics = 5

# Нам нужна сырая BOW-матрица с Шага 4 (train_bow_sparse, val_bow_sparse)
lda = LatentDirichletAllocation(n_components=n_components_topics, random_state=42, n_jobs=-1)

# Обучаем модель и трансформируем тексты в пропорции тем
train_lda_features = lda.fit_transform(train_bow_sparse)
val_lda_features = lda.transform(val_bow_sparse)

# Как посмотреть топ-слов для каждой темы
words = cv.get_feature_names_out() # берем имена слов из CountVectorizer
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[:-6:-1] # берем топ-5 слов
    top_words = [words[i] for i in top_words_idx]
    print(f"Тема #{topic_idx + 1}: {', '.join(top_words)}")

# Записываем как новые фичи в DataFrame
lda_cols = [f'LDA_Topic_{i+1}' for i in range(n_components_topics)]
train_lda_df = pd.DataFrame(train_lda_features, columns=lda_cols, index=X_train_lda.index)
val_lda_df = pd.DataFrame(val_lda_features, columns=lda_cols, index=X_val_lda.index)

**2. NMF (Non-negative Matrix Factorization / Неотрицательная матричная факторизация).** 

Чисто матричный подход. Он берет твою разреженную матрицу TF-IDF (размера $V \times D$, где $V$ - слова, $D$ - документы) и раскладывает её на произведение двух матриц меньшего размера: матрицы тем $W$ ($V \times K$) и матрицы весов документов $H$ ($K \times D$). Важное условие: все элементы матриц должны быть строго неотрицательными ($\ge 0$), благодаря чему компоненты легко интерпретировать (нет отрицательного вклада слов).

Работает значительно быстрее, чем LDA, так как сводится к оптимизации численных матричных функционалов. Часто дает более контрастные и четкие темы на небольших текстах.

Чувствителен к качеству предобработки TF-IDF.

In [ ]:
from sklearn.decomposition import NMF
import pandas as pd

X_train_nmf = X_train_text_clean.copy()
X_val_nmf = X_val_text_clean.copy()

# Для NMF берем TF-IDF матрицу с Шага 4 (train_tfidf_sparse, val_tfidf_sparse)
nmf = NMF(n_components=5, random_state=42, init='nndsvd')

train_nmf_features = nmf.fit_transform(train_tfidf_sparse)
val_nmf_features = nmf.transform(val_tfidf_sparse)

# Выводим топ-слов для интерпретации тем в NMF
tfidf_words = tfidf.get_feature_names_out()
for topic_idx, topic in enumerate(nmf.components_):
    top_words_idx = topic.argsort()[:-6:-1]
    top_words = [tfidf_words[i] for i in top_words_idx]
    print(f"Тема #{topic_idx + 1}: {', '.join(top_words)}")

# Записываем компоненты в DataFrame
nmf_cols = [f'NMF_Topic_{i+1}' for i in range(5)]
train_nmf_df = pd.DataFrame(train_nmf_features, columns=nmf_cols, index=X_train_nmf.index)
val_nmf_df = pd.DataFrame(val_nmf_features, columns=nmf_cols, index=X_val_nmf.index)

**Шаг девятый.** Обработка специфичных крайних случаев.

. В текстовых полях постоянно сидит скрытая структурированная информация: даты, номера документов, сокращения, имена или названия брендов. Если их просто прогнать через стандартную очистку из Шага 3, они превратятся в кашу или вовсе удалятся. Наша задача - извлечь из этих аномалий максимум пользы.

**1. NER (Named Entity Recognition - Распознавание именованных сущностей).**  Поиск и классификация особых сущностей в тексте: имен людей, названий организаций, гео-локаций (городов, стран) или денежных сумм.

Название города в тексте жалобы или имя VIP-персоны - это сильнейшие категориальные сигналы. Мы извлекаем их и создаем новые бинарные фичи (например, Contains_Location).

**2. Даты и числа внутри текста.** Поиск скрытых временных меток и численных параметров (например: "Договор от 12.04.2025", "ошибка 404").

Регулярные выражения вытаскивают эти данные, переводят в формат datetime или в отдельные числовые колонки до того, как Шаг 3 сотрет все цифры.

**3. Аббревиатуры и сокращения.** Приведение специфичного сленга и сокращений к единому словарному виду (например: "комп" $\to$ "компьютер", "универ" $\to$ "университет").

Помогает кодировщикам и эмбеддингам понять, что это одно и то же понятие.

**4. Многоязычный текст.** Корректная обработка, если в одной колонке перемешаны разные языки, путем разделения потоков или транслитерации.

In [ ]:
#!pip install natasha или spaCy

In [ ]:
import pandas as pd
import numpy as np
import re
from natasha import Segmenter, NewsEmbedding, NewsExtractor, Doc

X_train_spec = X_train_replace.copy()
X_val_spec = X_val_replace.copy()

text_col = 'Text_Column' # Твой текстовый столбец

# Инициализируем компоненты Natasha для NER
segmenter = Segmenter()
emb = NewsEmbedding()
extractor = NewsExtractor(emb)

# Монолитная функция для извлечения специфичных сущностей
def extract_specific_cases(df, col):
    # Гарантируем строковый тип
    texts = df[col].fillna("").astype(str).tolist()
    
    has_location = []
    has_person = []
    extracted_dates = []
    
    # Регулярное выражение для поиска дат (форматы: ДД.ММ.ГГГГ, ДД-ММ-ГГГГ, ГГГГ/ММ/ДД)
    date_pattern = re.compile(r'\b\d{2}[-./]\d{2}[-./]\d{4}\b|\b\d{4}[-./]\d{2}[-./]\d{2}\b')
    
    for text in texts:
        # --- 1. Обработка дат ---
        dates = date_pattern.findall(text)
        # Если нашли хотя бы одну дату, пишем 1, иначе 0
        extracted_dates.append(1 if dates else 0)
        
        # --- 2. Обработка NER (Имена и Города) ---
        if text.strip():
            doc = Doc(text)
            doc.segment(segmenter)
            doc.extract_ner(extractor)
            
            # Проверяем, какие типы сущностей нашел алгоритм в строке
            # LOC - локация (город, страна), PER - человек (имя, фамилия)
            types = [span.type for span in doc.spans]
            
            has_location.append(1 if 'LOC' in types else 0)
            has_person.append(1 if 'PER' in types else 0)
        else:
            has_location.append(0)
            has_person.append(0)
            
    # Записываем результаты в новые бинарные фичи
    df[f'{col}_has_date'] = extracted_dates
    df[f'{col}_has_loc'] = has_location
    df[f'{col}_has_person'] = has_person
    
    # --- 3. СЛОВАРЬ АББРЕВИАТУР (Пример раскрытия сокращений) ---
    abbrev_dict = {r'\bкпк\b': 'смартфон', r'\bб/у\b': 'подержанный', r'\bунив\b': 'университет'}
    for raw_bit, replacement in abbrev_dict.items():
        df[col] = df[col].str.replace(raw_bit, replacement, regex=True, flags=re.IGNORECASE)
        
    return df

# Применяем правила к трейну и валидации
X_train_spec = extract_specific_cases(X_train_spec, text_col)
X_val_spec = extract_specific_cases(X_val_spec, text_col)

print(X_train_spec[[text_col, f'{text_col}_has_date', f'{text_col}_has_loc', f'{text_col}_has_person']].head(3))

Подготовка данных полностью завершена! Остались только последние шаги: создать, обучить и оценить модель.

Разобьем модуль "моделирование" на 7 шагов:
1. Стратегия валидации
2. Метрики оценки моделей
3. Базовые классические модели
4. Продвинутые ансамбли и Бустинги
5. Глубокое обучение для таблиц
6. Тюнинг гиперпараметров
7. Финальное ансамблирование

**Шаг первый.** Стратегия валидации (Validation Strategy).

Это фундамент всего моделирования. Без надежной валидации модели обучаются "вслепую", постоянно натыкаясь на переобучение (overfitting) или получая завышенные ожидания.

**1. Hold-out split (Простое разбиение).** Стандарный метод.

Данные один раз статически делятся на две или три части: обучающую (Train) для подгонки весов, валидационную (Validation) для мониторинга качества и подбора параметров, и тестовую (Test) как финальный независимый судья. Обычно в пропорциях 70/15/15% или 80/20%.

Работает мгновенно, не требует повторного переобучения моделей.

Нестабильно, если выборка небольшая. Оценка качества критически зависит от того, какие именно строки случайно выпали в валидационный кусок.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Откусываем финальный тест (15% от всего датасета)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_sample, y_sample, test_size=0.15, random_state=42, stratify=y_sample
)

# 2. Оставшиеся 85% бьем на Train и Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.18, random_state=42, stratify=y_train_full
)

print(f"Hold-out размеры: Train {X_train.shape[0]}, Val {X_val.shape[0]}, Test {X_test.shape[0]}")

**2. K-Fold Cross-Validation (Классическая кросс-валидация).** Обычно самый надежный вариант.

Датасет бьется на K равных частей (фолдов). Модель обучается K раз: каждый раз K-1 фолдов идут на обучение, а оставшийся один fold - на валидацию. Финальная оценка - это среднее арифметическое по всем K итерациям.

Каждый объект датасета побывал в роли валидационного ровно 1 раз. Оценка метрики получается максимально стабильной и несмещенной.

Требует в K раз больше времени на вычисления.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

# Итерируемся по фолдам, получая индексы строк
for fold, (train_idx, val_idx) in enumerate(kf.split(X_sample)):
    X_tr, X_va = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_va = y_sample[train_idx], y_sample[val_idx]
    
    # Обучаем модель на текущем куске
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    
    # Считаем метрику
    score = accuracy_score(y_va, model.predict(X_va))
    fold_scores.append(score)
    print(f"Fold {fold+1} Accuracy: {score:.4f}")

print(f"Средняя Accuracy по K-Fold: {np.mean(fold_scores):.4f}")

**3. Stratified K-Fold (Стратифицированное разбиение).**

Полная копия K-Fold, но с жестким математическим контролем распределения целевой переменной. Внутри каждого фолда сохраняется ровно то же процентное соотношение классов, что и во всем исходном датасете (например, если в Spaceship Titanic выжило 50% и погибло 50%, то и в каждом фолде пропорция будет строго 50/50).

Жизненно необходим при дисбалансе классов (например, когда больных 1%, а здоровых 99%). Защищает от ситуации, когда в валидацию случайно попали одни нули.

Применим только для задач классификации (для непрерывной регрессии баланс долей не посчитать).

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
strat_scores = []

# Обязательно передаем y_sample в split(), чтобы он видел метки классов
for fold, (train_idx, val_idx) in enumerate(skf.split(X_sample, y_sample)):
    X_tr, X_va = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_va = y_sample[train_idx], y_sample[val_idx]
    
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    
    score = accuracy_score(y_va, model.predict(X_va))
    strat_scores.append(score)
    print(f"Fold {fold+1} (Доля 1 класса в Val: {np.mean(y_va):.2f}) -> Accuracy: {score:.4f}")

**4. GroupKFold (Групповое разбиение).**

Используется, когда в данных есть зависимые группы объектов (например, несколько строк принадлежат одному и тому же пациенту, или в Spaceship Titanic - члены одной семьи по префиксу PassengerId). Алгоритм гарантирует, что объекты из одной группы никогда не разделятся между train и validation. Группа целиком улетает либо в обучение, либо в проверку.

Железная защита от скрытых утечек информации (Data Leakage) и завышенных метрик.

Требует наличия явного столбца с идентификатором групп.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression

gkf = GroupKFold(n_splits=5)

# В split() обязательно передаем массив или колонку с ID групп
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_sample, y_sample, groups=groups_sample)):
    X_tr, X_va = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_va = y_sample[train_idx], y_sample[val_idx]
    
    # Проверяем, что группы из вала не пересекаются с трейном
    train_groups = set(groups_sample[train_idx])
    val_groups = set(groups_sample[val_idx])
    assert len(train_groups.intersection(val_groups)) == 0, "Утечка группы!"
    
    print(f"Fold {fold+1}: Успешно изолирован. Объектов в Val: {len(X_va)}")

**5. Leave-One-Out (LOO).**

Экстремальный случай кросс-валидации, когда K равняется общему числу строк в датасете (N). Модель обучается на N-1 объектах, а предсказывает для одной-единственной оставленной строчки. Процесс повторяется N раз.

Идеально выжимает стабильную оценку, когда данных критически мало (например, медицинский датасет на 30–50 пациентов).

Компьютер вычислительно умрет, если натравить LOO на датасет крупнее нескольких сотен строк.

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression

loo = LeaveOneOut()
loo_scores = []

# Будет столько итераций, сколько всего строк в датасете
for train_idx, val_idx in loo.split(X_sample):
    X_tr, X_va = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_va = y_sample[train_idx], y_sample[val_idx]
    
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    
    # Записываем 1, если угадали единственный объект, иначе 0
    pred = model.predict(X_va)
    loo_scores.append(1 if pred[0] == y_va[0] else 0)

print(f"Итоговая точность на LOO: {np.mean(loo_scores):.4f}")

**6. TimeSeriesSplit (Валидация по времени).**

Валидация «скользящим окном» для временных рядов или хронологических данных. Прошлое может предсказывать будущее, но будущее никогда не должно учить прошлое. На итерации i модель учится на данных до момента времени T, а валидируется на отрезке $T + \Delta t$.

Исключает заглядывание в будущее.

Размер обучающей выборки на первых фолдах может быть слишком маленьким.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression

tscv = TimeSeriesSplit(n_splits=4)

# Окно обучения постепенно расширяется сверху вниз
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sample)):
    X_tr, X_va = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_va = y_sample[train_idx], y_sample[val_idx]
    
    print(f"Fold {fold+1}: Учимся на индексах 0..{max(train_idx)} -> Валидируемся на {min(val_idx)}..{max(val_idx)}")

**Шаг второй.** Метрики оценки моделей (Evaluation Metrics).

Их стоит разделять по двум классам бинарным предсказаням и предсказание числа.

**Задача классификации (бинарной).**

1. **Accuracy (Доля правильных ответов).** Отношение числа верных предсказаний к общему числу объектов: $\frac{TP + TN}{TP + TN + FP + FN}$. Абсолютно бесполезна при дисбалансе классов. Если в банке 99% клиентов возвращают кредит, то "тупой" классификатор, всегда предсказывающий 1, получит Accuracy = 99%, но не найдет ни одного мошенника.
2. **Precision (Точность).** Доля реальных объектов положительного класса среди всех, кого модель отметила как положительные: $\frac{TP}{TP + FP}$. Минимизирует ложноположительные срабатывания ($FP$).
3. **Recall (Полнота / Чувствительность / TPR).** Доля найденных моделью объектов положительного класса среди всех реальных объектов этого класса: $\frac{TP}{TP + FN}$. Минимизирует пропуски ($FN$). Например, медицинская диагностика рака или детекция опасных поломок на производстве. Важно найти всех больных, пускай даже ценой ложной тревоги.
4. **$F_1$-score ($F$-мера).** Гармоническое среднее между Precision и Recall: $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$.Зачем нужна: Позволяет оценить баланс между точностью и полнотой одной цифрой. За счет гармонического среднего метрика жестко штрафует модель, если хотя бы один из показателей проваливается в ноль.
5. **ROC-AUC (Площадь под ROC-кривой).** Метрика качества ранжирования объектов по вероятностям. Показывает вероятность того, что случайно выбранный объект класса 1 получит оценку выше, чем случайно выбранный объект класса 0.Плюсы: Устойчива к дисбалансу классов и не привязана к жесткому порогу вероятности (0.5).
6. **Log Loss (Логарифмическая функция потерь).** Жестко штрафует модель за уверенность в неверном ответе. Требует от алгоритма выдавать идеально калиброванные вероятности.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, log_loss, confusion_matrix, classification_report, ConfusionMatrixDisplay
)

# Предположим, это реальные метки и предсказанные вероятности вашей модели на валидации
y_true_cls = np.array([0, 1, 1, 0, 1, 0, 0, 1, 0, 1])
y_pred_proba = np.array([0.1, 0.9, 0.8, 0.3, 0.4, 0.2, 0.6, 0.7, 0.1, 0.85])

# Переводим вероятности в жесткие метки классов по стандартному порогу 0.5
y_pred_labels = (y_pred_proba >= 0.5).astype(int)

print("--- МЕТРИКИ КЛАССИФИКАЦИИ ---")
print(f"Accuracy (Доля правильных): {accuracy_score(y_true_cls, y_pred_labels):.4f}")
print(f"Precision (Точность):       {precision_score(y_true_cls, y_pred_labels):.4f}")
print(f"Recall (Полнота):           {recall_score(y_true_cls, y_pred_labels):.4f}")
print(f"F1-Score (F-мера):          {f1_score(y_true_cls, y_pred_labels):.4f}")
print(f"ROC-AUC (Качество ранж.):   {roc_auc_score(y_true_cls, y_pred_proba):.4f}")
print(f"Log Loss (Штраф за увер.):  {log_loss(y_true_cls, y_pred_proba):.4f}")

# Подробный текстовый отчет по классам (0 и 1 отдельно)
print(classification_report(y_true_cls, y_pred_labels))

# Визуализация Confusion Matrix (Матрицы ошибок)
cm = confusion_matrix(y_true_cls, y_pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap='Blues')
plt.title("Матрица ошибок (Confusion Matrix)")
plt.show()

**Задача регрессии (предсказание числа).**
1. **MAE (Mean Absolute Error - Средняя абсолютная ошибка).** Среднее арифметическое модулей отклонений: $\frac{1}{n}\sum|y_i - \hat{y}_i|$. Физически понятна (ошибка измеряется в тех же единицах, что и таргет: в рублях, градусах). Устойчива к выбросам (робастна). При оптимизации MAE модель сходится к медиане распределения.
2. **MSE (Mean Squared Error - Среднеквадратичная ошибка).** Среднее квадратов отклонений: $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$. Квадрат дико штрафует за крупные ошибки. Если модель ошиблась на 2 рубля - штраф 4, а если на 10 рублей - штраф 100. Неустойчива к выбросам (одна аномальная строка может исказить всё MSE). При оптимизации MSE модель сходится к среднему арифметическому.
3. **RMSE (Root Mean Squared Error).** Квадратный корень из MSE: $\sqrt{\text{MSE}}$. Возвращает метрику из квадратов обратно в физическую размерность целевой переменной.
4. **$R^2$ (Коэффициент детерминации).** Показывает долю дисперсии таргета, объясненную моделью: $1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{baseline}}}$. Если $R^2 = 1$ - модель идеальна. Если $R^2 = 0$ - модель работает на уровне предсказания простого среднего значения. Если $R^2 < 0$ - модель работает хуже, чем константное среднее (код сломан).
5. **MAPE (Mean Absolute Percentage Error).** Средняя относительная ошибка в процентах. Измеряет отклонение относительно масштаба самого объекта. Удобна для бизнеса.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# Исходные цены объектов и предсказания вашей регрессионной модели
y_true_reg = np.array([100.0, 150.0, 300.0, 500.0, 200.0])
y_pred_reg = np.array([110.0, 140.0, 290.0, 420.0, 205.0])

print("--- МЕТРИКИ РЕГРЕССИИ ---")
print(f"MAE (Средняя абсолютная):            {mean_absolute_error(y_true_reg, y_pred_reg):.4f}")
print(f"MSE (Среднеквадратичная):            {mean_squared_error(y_true_reg, y_pred_reg):.4f}")
print(f"RMSE (Корень из среднеквадр.):       {np.sqrt(mean_squared_error(y_true_reg, y_pred_reg)):.4f}")
print(f"R² (Коэффициент детерминации):       {r2_score(y_true_reg, y_pred_reg):.4f}")
print(f"MAPE (Относительная ошибка в %):    {mean_absolute_percentage_error(y_true_reg, y_pred_reg) * 100:.2f}%")

**Шаг третий.** Базовые классические модели (Baselines & Simple Models).

1. **Dummy Classifier (Константный Baseline).**

Абсолютно "тупая" модель. Она не смотрит на признаки, а просто предсказывает самый частый класс (моду) или среднее значение. Если твоя сложная модель выдает метрику хуже или наравне с Dummy, значит, твоя предобработка сломана или модель переобучилась.

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# strategy="most_frequent" означает всегда предсказывать самый частый класс из train
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_preds = dummy.predict(X_val)
# Для ROC-AUC нужны предсказанные вероятности (второй столбец)
dummy_probas = dummy.predict_proba(X_val)[:, 1]

print("DUMMY BASELINE")
print(f"Accuracy:  {accuracy_score(y_val, dummy_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, dummy_probas):.4f}") 
# Важно: ROC-AUC константной модели всегда будет равен 0.5 (уровень случайного угадывания)

2. **kNN (k-Ближайших соседей).** 

Классический метрический алгоритм. Принимает решение на основе классов самых близких к нему объектов из обучающей выборки.

kNN критически требует масштабирования данных (Scaling) на этапе предобработки, иначе признаки с большими значениями (например, доход или цена) полностью подавят бинарные фичи при подсчете расстояний.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Создаем kNN с 5 соседями
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train, y_train)

knn_preds = knn.predict(X_val)
knn_probas = knn.predict_proba(X_val)[:, 1]

print("kNN CLASSIFIER")
print(f"Accuracy:  {accuracy_score(y_val, knn_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, knn_probas):.4f}")

3. **Линейные модели: Logistic Regression & Regularization (Ridge / Lasso).**

Проверяет наличие линейных зависимостей в данных. В классификации используется логистическая регрессия. Чтобы модель не переобучалась на избыточных фичах, применяют штрафы (регуляризацию).

penalty='l2' (Ridge / Тихоновская): Квадрат весов. Мягко и пропорционально уменьшает все веса признаков к нулю, но не зануляет их полностью.

penalty='l1' (Lasso): Абсолютное значение весов. Работает как Feature Selection - жестко зануляет веса у бесполезных признаков, оставляя только самые сильные.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Логистическая регрессия с L2 регуляризацией (Ridge)
# Параметр C - инверсия силы регуляризации (чем меньше C, тем сильнее штраф)
log_reg_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
log_reg_l2.fit(X_train, y_train)

print("LOGISTIC REGRESSION (L2)")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, log_reg_l2.predict_proba(X_val)[:, 1]):.4f}")

# 2. Логистическая регрессия с L1 регуляризацией (Lasso - отбор признаков)
# solver='liblinear' необходим для поддержки L1-штрафа в sklearn
log_reg_l1 = LogisticRegression(penalty='l1', C=0.5, solver='liblinear', random_state=42)
log_reg_l1.fit(X_train, y_train)

print("\nLOGISTIC REGRESSION (L1)")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, log_reg_l1.predict_proba(X_val)[:, 1]):.4f}")
# Лайфхак: можно посмотреть, сколько признаков модель полностью занулила
zero_weights = np.sum(log_reg_l1.coef_ == 0)
print(f"Lasso занулила {zero_weights} из {X_train.shape[1]} признаков.")

4. **Decision Tree (Одиночное дерево решений).**

Строит логическую цепочку условий (сплитов) вида "Если возраст > 18 и траты на Spa == 0...".

Одиночное дерево без ограничений глубины мгновенно запоминает обучающую выборку на 100%, превращаясь в чистую переобученную шпаргалку. Поэтому всегда жестко фиксируем max_depth.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Ограничиваем максимальную глубину (max_depth) и минимальное число объектов для сплита (min_samples_split)
tree = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
tree.fit(X_train, y_train)

tree_preds = tree.predict(X_val)
tree_probas = tree.predict_proba(X_val)[:, 1]

print("DECISION TREE")
print(f"Accuracy:  {accuracy_score(y_val, tree_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, tree_probas):.4f}")

**Шаг четвертый.** Продвинутые ансамбли и градиентные бустинги (Ensembles & Boosting).

1. **Bagging (Бэггинг) и Random Forest (Случайный лес).**

Обучает много глубоких деревьев независимо и параллельно на случайных подвыборках данных (метод Bootstrap). Финальный прогноз - это просто среднее арифметическое (для регрессии) или голосование большинством (для классификации).

Random Forest: Улучшенный бэггинг. При построении каждого сплита в дереве выбирается случайное подмножество из всего max_features признаков. Это снижает коррелированность деревьев между собой.

Бэггинг снижает дисперсию (variance) сложного алгоритма, но не меняет его смещение (bias). Случайный лес практически невозможно переобучить увеличением количества деревьев (n_estimators).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# n_estimators - количество деревьев
# max_features='sqrt' - брать случайные фичи для снижения корреляции
rf = RandomForestClassifier(n_estimators=100, max_depth=8, max_features='sqrt', n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

print("RANDOM FOREST")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1]):.4f}")

2. **Градиентный бустинг (Gradient Boosting).**

Деревья обучаются последовательно. Каждое новое дерево обучается предсказывать не сам таргет, а антиградиент (остатки ошибки) предыдущей композиции деревьев.

Бустинг снижает смещение (bias) простых алгоритмов. Из-за этого бустинг очень легко переобучить, если вовремя не остановиться (early_stopping_rounds) или взять слишком большой темп обучения (learning_rate).

Три главных библиотеки бустинга:
* XGBoost: Каноничная библиотека, использующая разложение Тейлора второго порядка для функции потерь и сильную встроенную регуляризацию структуры дерева.

* LightGBM: Скоростной бустинг от Microsoft. Растит деревья не по уровням (Level-wise), а по листьям (Leaf-wise), выбирая лист с максимальной потерей. Использует гистограммный подход для непрерывных фичей.

* CatBoost: Библиотека от Яндекса. Строит регулярные (симметричные) деревья, что дает колоссальную скорость инференса и устойчивость к переобучению. Содержит встроенные продвинутые схемы кодирования категорий (Target Encoding) прямо во время построения сплитов.

In [1]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Для бинарной классификации используем objective='binary:logistic'
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,         # доля строк для обучения одного дерева
    colsample_bytree=0.8,  # доля признаков для одного дерева
    random_state=42,
    n_jobs=-1
)

# Обучаем с ранней остановкой, чтобы избежать переобучения
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("XGBOOST")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1]):.4f}")

3250


In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,         # LightGBM контролирует размер дерева через листья
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)

print("LIGHTGBM")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, lgb_model.predict_proba(X_val)[:, 1]):.4f}")

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

cb_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,         # L2 регуляризация значений в листьях
    random_seed=42,
    verbose=False
)

cb_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=30
)

print("CATBOOST")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, cb_model.predict_proba(X_val)[:, 1]):.4f}")

**Шаг пятый.** Глубокое обучение для табличных данных (Neural Networks).

Хотя градиентный бустинг и случайный лес являются безоговорочными королями классических таблиц, в современной практике машинного обучения полносвязные нейросети и специализированные глубокие архитектуры часто используются как сильные альтернативные кандидаты. Они добавляют необходимое разнообразие в ансамбли и отлично улавливают сложные скрытые нелинейные зависимости (особенно если в датасете присутствуют извлеченные текстовые эмбеддинги).

1. MLP (Multilayer Perceptron / Многослойный перцептрон)
2. TabNet (Специализированная табличная архитектура)

-не будем расписывать подробнее, так как это уже уходит далеко за наш уровень

**Шаг шестой.** Тюнинг гиперпараметров (Hyperparameter Optimization).

После того как мы выбрали метрики и обучили базовые варианты моделей, нам нужно выжать из них максимум, «покрутив» внутренние настройки (гиперпараметры). Для этого используются три принципиально разных подхода:

1. **GridSearchCV (Поиск по сетке).** Полный детерминированный перебор всех возможных комбинаций параметров, которые указаны в словаре.

Гарантированно найдет наилучшую комбинацию из предложенных.

Тратит колоссально много времени.

In [ ]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

# 1. Задаем жесткую сетку параметров
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200]
}

lgb_grid = lgb.LGBMClassifier(random_state=42, verbosity=-1)

# 2. Инициализируем поиск (под капотом автоматически работает Кросс-валидация cv=3)
grid_search = GridSearchCV(
    estimator=lgb_grid, 
    param_grid=param_grid, 
    scoring='roc_auc', 
    cv=3, 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("GRID SEARCH CV")
print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший Train ROC-AUC: {grid_search.best_score_:.4f}")

2. **RandomizedSearchCV (Случайный поиск).** Вместо полного перебора алгоритм выбирает случайные комбинации параметров из заданных распределений фиксированное количество раз (n_iter).

Работает в разы быстрее, при этом математически доказано, что качество подбора почти не уступает полному перебору по сетке.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import lightgbm as lgb

# 1. Вместо списков можем задавать непрерывные распределения
param_dist = {
    'max_depth': randint(4, 10),
    'learning_rate': uniform(0.01, 0.2),
    'n_estimators': randint(100, 500)
}

lgb_rand = lgb.LGBMClassifier(random_state=42, verbosity=-1)

# 2. n_iter=10 означает, что алгоритм проверит ровно 10 случайных комбинаций
random_search = RandomizedSearchCV(
    estimator=lgb_rand, 
    param_distributions=param_dist, 
    n_iter=10, 
    scoring='roc_auc', 
    cv=3, 
    n_jobs=-1, 
    random_state=42
)

random_search.fit(X_train, y_train)

print("RANDOMIZED SEARCH CV")
print(f"Лучшие параметры: {random_search.best_params_}")
print(f"Лучший Train ROC-AUC: {random_search.best_score_:.4f}")

3. **Optuna (Байесовская оптимизация - Современный стандарт).** Умный подбор. Каждая следующая итерация учитывает результаты предыдущих. Оптимизатор строит суррогатную вероятностную модель и целенаправленно идет в ту область пространства параметров, где метрика росла.

Идеально подходит для тяжелых моделей (XGBoost, CatBoost, Нейросети). Экономит время и находит скрытые экстремумы. Умеет отрезать заведомо провальные попытки на ходу (pruning).

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import cross_val_score

# Подавляем лишние логи самой Optuna для чистоты вывода
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Описываем целевую функцию (Objective), которую Optuna будет максимизировать
def objective(trial):
    # Задаем пространство поиска с помощью trial-методов
    params = {
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'num_leaves': trial.suggest_int('num_leaves', 20, 128),
        'random_state': 42,
        'verbosity': -1
    }
    
    clf = lgb.LGBMClassifier(**params)
    
    # Оцениваем параметры с помощью кросс-валидации на Train выборке
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    return score

# 2. Запускаем умный процесс оптимизации
# direction="maximize", так как мы хотим поднять ROC-AUC как можно выше
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15) # n_trials - сколько шагов сделать алгоритму

print("OPTUNA OPTIMIZATION")
print(f"Лучшие параметры: {study.best_params}")
print(f"Лучший Train ROC-AUC: {study.best_value:.4f}")

**Шаг седьмой.** Ансамблирование моделей (Meta-Ensembling).

Когда базовые классические модели и бустинги уже идеально настроены и затюнены на Шаге 6, одиночный алгоритм упирается в свой технологический потолок. Чтобы пробить его и выжать максимум точности, предсказания лучших моделей объединяют в единый ансамбль.

1. **Voting (Голосование моделей).** Самый простой способ подружить модели из коробки. Бывает двух типов:

* Hard Voting (Жесткое голосование). Каждый алгоритм голосует за конкретный класс (0 или 1). Финальный ответ выбирается простым большинством голосов.
* Soft Voting (Мягкое голосование). Модели возвращают предсказанные вероятности классов. Эти вероятности усредняются (обычно как среднее арифметическое). Мягкое голосование работает точнее, так как учитывает степень "уверенности" каждого алгоритма.

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Инициализируем затюненные на Шаге 6 модели
model1 = CatBoostClassifier(iterations=300, random_seed=42, verbose=False)
model2 = lgb.LGBMClassifier(n_estimators=300, random_state=42, verbosity=-1)
model3 = xgb.XGBClassifier(n_estimators=300, random_state=42, eval_metric='logloss')

# Создаем ансамбль голосования. voting='soft' заставит усреднять вероятности
voting_clf = VotingClassifier(
    estimators=[('cb', model1), ('lgb', model2), ('xgb', model3)],
    voting='soft',
    n_jobs=-1
)

voting_clf.fit(X_train, y_train)
voting_probas = voting_clf.predict_proba(X_val)[:, 1]

print("SOFT VOTING ENSEMBLE")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, voting_probas):.4f}")

2. **Blending (Блендинг).** Метод взвешенного усреднения. Вместо простого среднего мы подбираем веса (важность) для предсказаний каждой модели:
$$Y_{blend} = w_1 \cdot P_{catboost} + w_2 \cdot w_{lightgbm} + w_3 \cdot P_{mlp}$$

 Чтобы не переобучить веса $w_i$, подбор коэффициентов делают строго на отдельной отложенной hold-out выборке (которую не видел ни один базовый алгоритм во время обучения).

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Обучаем базовые модели на X_train
model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

# 2. Получаем их предсказания на валидационной (отложенной) выборке
p1 = model1.predict_proba(X_val)[:, 1]
p2 = model2.predict_proba(X_val)[:, 1]
p3 = model3.predict_proba(X_val)[:, 1]

# 3. Задаем веса вручную (сумма весов должна быть равна 1.0)
# Логика: даем больший вес той модели, у которой локальная метрика была выше
weights = [0.5, 0.3, 0.2] 

blend_probas = weights[0] * p1 + weights[1] * p2 + weights[2] * p3

print("BLENDING ENSEMBLE")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, blend_probas):.4f}")

3. **Stacking (Стекинг).** Продвинутый подход, где веса или логика объединения базовых моделей настраиваются не руками, а отдельной мета-моделью (чаще всего берут простую логистическую регрессию или Ridge, чтобы избежать переобучения).

Если базовые модели сделают предсказания для того же train, на котором учились, мета-модель моментально поймет, кто из них зазубрил ответы, и переобучится.

Датасет бьется на фолды. Базовые модели обучаются на $K-1$ фолдах и предсказывают для оставшегося фолда. Этот процесс циклически повторяется, пока мы не соберем "чистые" (Out-of-Fold) предсказания для абсолютно каждой строчки train. Именно на этих OOF-предсказаниях как на новых фичах и обучается мета-модель.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# В качестве мета-модели берем классическую логистическую регрессию с L2-регуляризацией
meta_allocator = LogisticRegression(penalty='l2', C=1.0, random_state=42)

# cv=5 означает, что базовые модели будут строить OOF-предсказания по 5 фолдам
stacking_clf = StackingClassifier(
    estimators=[('cb', model1), ('lgb', model2), ('xgb', model3)],
    final_estimator=meta_allocator,
    cv=5,
    n_jobs=-1,
    passthrough=False # False означает, что мета-модель увидит ТОЛЬКО предсказания базовых моделей
)

print("Запуск Stacking пайплайна (включает построение OOF по кросс-валидации)...")
stacking_clf.fit(X_train, y_train)
stacking_probas = stacking_clf.predict_proba(X_val)[:, 1]

print("\nSTACKING ENSEMBLE")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, stacking_probas):.4f}")